# Climate disclosure classification
## Stakeholder briefing
A sustainability analyst may need to read hundreds of company disclosures to find statements about climate risks and business opportunities. This prototype puts paragraphs into three reading categories: **risk**, **neutral**, and **opportunity**. These are the dataset's categories for disclosure wording; they are not a certification that a company is environmentally responsible. This boundary follows the [dataset description](data/SOURCE_README.md), which describes labelled corporate-report paragraphs rather than independently verified environmental outcomes.

**The recommendation is to use the prototype only with a person reviewing its suggestions.** In the controlled test, the transformer matches the dataset label for 243 of 320 paragraphs, compared with 228 for the classical model. However, it still misses 24 of the 51 opportunity paragraphs and 15 of the 106 risk paragraphs. Those misses matter if a team uses the categories to decide what can be ignored. The counts are reconstructed from the saved predictions in Section 7; the class-by-class results and examples in Sections 8–10 explain the decision.

The project also asks for a balanced quality score of at least **0.80** before a model can qualify for the proposed pilot. “Balanced” means giving equal attention to each category, including opportunity, which is the least common category in this test. The transformer reaches **0.719** and the classical model **0.660**. A score of 0.719 does **not** mean that exactly 71.9% of paragraphs are correct: the actual correct share is 75.9%. It is a separate measure that penalizes missed labels and false alarms in each category. Section 7.2 walks through that distinction using the actual counts.

The transformer is therefore a better candidate for further development, but the evidence does not support replacing the analyst. It corrects 37 classical errors while introducing 22 new ones, and no tested validation threshold achieves both the proposed accuracy and coverage targets for automatic acceptance. These are measured limits of this experiment, not evidence that the tool has no possible value: a supervised pilot could test whether its suggestions help people find relevant passages faster, but that time-saving benefit has not been measured. Evidence: Sections 8, 10 and 11; the threshold and review-workload tables in Sections 5 and 7.1.

*Evidence scope:* fixed numerical interpretations refer to controlled run `20261102T152220909909Z_3d7d7c00a8`. Review mode recomputes that comparison. A newly trained experiment must be interpreted from its own generated tables and decision; the fixed examples are not an automatic claim about a different fitted model.

## 1. Decision, success criteria and scope
The intended user is a sustainability research team deciding which disclosure paragraphs require attention. **Use-case assumption:** overlooking a labelled risk paragraph is more consequential than sending an additional paragraph to an analyst. No stakeholder cost study was supplied, so this is an explicit design assumption, not an observed business preference. It motivates a separate risk-recall gate alongside balanced classification quality.

**Proposed project gates, not universal safety standards:** macro F1 ≥ 0.80; risk recall ≥ 0.85; warmed-up batch inference ≤100 ms per paragraph on the recorded hardware. A transformer must additionally improve paired test macro F1 by ≥0.03, with the 95% paired bootstrap interval above zero and no drop in risk recall. A validation-selected automatic-acceptance threshold must retain ≥50% of paragraphs, achieve ≥95% observed accuracy on retained paragraphs, and retain at least 30 validation examples before test confirmation. The exact 0.80 value has not been calibrated by a stakeholder cost study, so it is not claimed to be optimal. These numerical targets are assumptions in [the configuration](configs/run_config.json); they need stakeholder approval and evaluation on the actual workload. Missing a gate means this report recommends human-led use rather than automatic acceptance.

The categories are **risk, neutral and opportunity** as defined by the [source card](data/SOURCE_README.md). Neutral is not evidence of safety, and opportunity is not evidence that an environmental claim is true. The dataset contains English paragraphs, so this evaluation does not establish performance for other languages or complete company reports. No issuer-disjoint or temporal test is possible from the supplied `text` and `label` fields; shared issuers and repeated wording could make the paragraph-level comparison look more general than it is. Exact-duplicate and similarity diagnostics are reported in Sections 3 and 8.

Macro F1 is the main selection metric because it weights the three categories equally. Accuracy reports the overall correct share, while risk recall reports the fraction of labelled risk paragraphs found. Average precision is the mean of one-vs-rest average-precision scores, not trapezoidal PR-AUC. Model probability outputs are treated as scores rather than calibrated probabilities of correctness: Section 9 includes actual high-score errors, and Section 5 separately checks whether thresholding meets the proposed operational targets. [Metric definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html).

**Validity limit:** the test set is used for reporting and error inspection, not model selection. Any changes proposed from its errors must be evaluated on a fresh holdout. Otherwise the comparison would no longer test generalization to unexamined paragraphs. Qualitative interpretations below are examples and hypotheses, not proof that dataset labels are wrong or that a particular training change will help.

## 2. Reproduce the evidence or train an experiment
**Review the saved comparison without a GPU:** install [requirements.txt](requirements.txt), then run `python scripts/run_notebook.py --mode review`. This executes the notebook from a clean kernel using both model families' saved test scores, the retained classical pipeline, exact split IDs, training logs and source tables from the controlled Colab run. It recomputes metrics, bootstrap intervals, explanations and charts without claiming to train the transformer again. `NLP_REVIEW_RUN` can select a different compatible run directory.

**Train on Colab:** install [requirements-transformer.txt](requirements-transformer.txt), select a CUDA GPU and use `--mode full` in the runner or `NLP_RUN_MODE=full` in the notebook. With Google's VS Code extension use Select Kernel → Colab → Auto Connect; in browser Colab upload this notebook, run the bootstrap, restart after package installation, then Run all. Setup and artifact layout are in [README.md](README.md). Full mode requires CUDA; baseline mode deliberately skips transformer training. A standalone upload defaults to training/analysis when the saved review directory is absent.

The exact data/model commits, package specifications, seed list and hyperparameters are recorded with each run. Seeds reduce variation but do not guarantee identical refits across devices. To reproduce the reported classical predictions, `scripts/replay_classical.py --run artifacts/20261102T152220909909Z_3d7d7c00a8 --refit` freezes the saved training-only vocabulary before refitting; all 320 predictions have been checked this way. This addresses ambiguity among equally frequent terms at the 1,000-feature limit without choosing features from test text. GPU seed variation is reported separately in Section 7.

**Artifact limit:** transformer tensors were not retained locally after the Colab contents service rejected the 2.54 GB archive. The saved predictions, logs, tokenizers, configuration and expected file hashes are local; current remote availability is unverified. Review mode therefore supports analytical reproduction and classical explanations, not local transformer re-inference or transformer token attributions. A GPU training run regenerates the tensors. [Exact collection record](artifacts/20261102T152220909909Z_3d7d7c00a8/collection_status.json).

In [ ]:
import os, sys, json, subprocess, importlib.metadata
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
ROOT = Path.cwd()
CORE_REQUIREMENTS = '# Python 3.13.5 reference; Python 3.12/3.13 supported (record exact runtime per run).\nnumpy==2.1.3\npandas==2.2.3\nscipy==1.15.3\nscikit-learn==1.6.1\nmatplotlib==3.10.0\npyarrow==19.0.0\njoblib==1.4.2\nnbformat==5.10.4\nnbclient==0.10.2\nipykernel==6.29.5\nnbconvert==7.16.6\n'
GPU_REQUIREMENTS = "-r requirements.txt\n# Separate GPU profile; no transformer training is started on CPU by default.\ntorch==2.8.0\ntransformers==4.57.1\naccelerate==1.11.0\nhuggingface-hub==0.36.0\ntokenizers==0.22.1\nsafetensors==0.6.2\n# Keep Colab's optional vision/audio packages ABI-compatible with torch.\ntorchvision==0.23.0\ntorchaudio==2.8.0\n"

if IN_COLAB:
    (ROOT / 'requirements.txt').write_text(CORE_REQUIREMENTS)
    (ROOT / 'requirements-transformer.txt').write_text(GPU_REQUIREMENTS)
    wanted = [line for line in (CORE_REQUIREMENTS + GPU_REQUIREMENTS).splitlines()
              if '==' in line and not line.startswith('#')]
    mismatch = []
    for spec in wanted:
        package, version = spec.split('==')
        try:
            current = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            current = None
        if current != version:
            mismatch.append(spec)
    if mismatch:
        install=subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                                '-r', str(ROOT / 'requirements-transformer.txt')],capture_output=True,text=True)
        if install.returncode:
            print(install.stdout[-10000:])
            print(install.stderr[-10000:])
            install.check_returncode()
        raise RuntimeError('Pinned packages installed. Restart the Colab session, then Run all.')
print('Runtime:', 'Colab' if IN_COLAB else 'local / remote Jupyter', sys.version.split()[0])

### 2.1 Configuration and run record
Each execution creates its own `artifacts/<UTC timestamp>_<config hash>/`. It saves the effective config, package snapshot, hardware, splits, model files, predictions, tables, figures and a checksum manifest. A complete manifest is written only in the final cell. `latest_run.json` points to that run. The environment variable `NLP_CONFIG_PATH` selects an alternative config; the effective copy is always saved.

In [ ]:
import hashlib, platform, random, re, time, shutil, zipfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score, average_precision_score,
                             confusion_matrix, classification_report)
from sklearn.preprocessing import label_binarize
import joblib
DEFAULT_CONFIG = {'protocol_version': '2.0', 'seed': 42, 'validation_fraction': 0.2, 'dataset_revision': 'f261b1fd1af11195db3bdfd809a37a85e0d9fbc2', 'model_id': 'distilbert/distilbert-base-uncased', 'model_revision': '12040accade4e8a0f71eabdb258fecc2e7e948be', 'baseline': {'min_df': 5, 'max_features': 1000, 'C_grid': [0.1, 1.0, 10.0], 'alpha': 1.0, 'max_iter': 1000}, 'transformer': {'epochs': 3, 'learning_rate': 2e-05, 'batch_size': 16, 'max_length': 128, 'weight_decay': 0.01, 'warmup_ratio': 0.0, 'fp16': False, 'early_stopping_patience': 1, 'seeds': [42, 43, 44]}, 'bootstrap_resamples': 2000, 'decision': {'min_macro_f1': 0.8, 'min_risk_recall': 0.85, 'min_transformer_gain': 0.03, 'min_retained_accuracy': 0.95, 'min_coverage': 0.5, 'min_validation_retained_n': 30, 'max_inference_ms_per_paragraph': 100.0}, 'routing_threshold_grid': [0.5, 0.6, 0.7, 0.8, 0.9, 0.95], 'run_transformer': 'auto_cuda_only', 'run_pos': False}
DATA_MANIFEST = {'dataset': 'climatebert/climate_sentiment', 'revision': 'f261b1fd1af11195db3bdfd809a37a85e0d9fbc2', 'files': {'test-00000-of-00001-3f9f7af4f5914b8e.parquet': {'sha256': '47bc8ce57d5d975628063362827543df9382511e1a6d501896bf3a678a36fc75', 'bytes': 100977}, 'train-00000-of-00001-04b49ae22f595095.parquet': {'sha256': '709f6dd5b265e9de31357c112ce7673ee3522e364e7e0712c649216f83a4499b', 'bytes': 272661}}, 'label_names': ['risk', 'neutral', 'opportunity'], 'license': 'CC-BY-NC-SA-4.0', 'source_card': 'https://huggingface.co/datasets/climatebert/climate_sentiment/blob/f261b1fd1af11195db3bdfd809a37a85e0d9fbc2/README.md'}

config_path = Path(os.environ.get('NLP_CONFIG_PATH', ROOT / 'configs/run_config.json'))
CONFIG = json.loads(config_path.read_text()) if config_path.exists() else DEFAULT_CONFIG.copy()
assert CONFIG['dataset_revision'] == DATA_MANIFEST['revision'], 'Update the checksummed data manifest with a new revision.'
REVIEW_DIR = Path(os.environ.get('NLP_REVIEW_RUN', ROOT / 'artifacts/20261102T152220909909Z_3d7d7c00a8'))
MODE = os.environ.get('NLP_RUN_MODE', 'review' if REVIEW_DIR.exists() else 'auto')
assert MODE in ('auto', 'baseline', 'full', 'review')
REVIEW_MODE = MODE == 'review'
if REVIEW_MODE:
    assert (REVIEW_DIR/'manifest.json').exists(), 'Review mode needs a collected comparison run.'
    recorded_config=json.loads((REVIEW_DIR/'run_config.json').read_text())
    for key in ['seed','validation_fraction','dataset_revision','model_id','model_revision','baseline','transformer','bootstrap_resamples','decision']:
        CONFIG[key]=recorded_config[key]
    SEED=CONFIG['seed']
CONFIG['execution_mode'] = MODE
CONFIG['run_pos'] = os.environ.get('NLP_RUN_POS', str(CONFIG['run_pos'])).lower() == 'true'
SEED = CONFIG['seed']
random.seed(SEED)
np.random.seed(SEED)
LABELS = [0, 1, 2]
NAMES = ['risk', 'neutral', 'opportunity']
COLORS = ['#b54736', '#526b80', '#167c70']
plt.rcParams.update({'figure.figsize': (10, 4.8), 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'savefig.facecolor': 'white', 'axes.labelcolor': '#202830', 'text.color': '#202830'})
pd.set_option('display.max_colwidth', 110)
config_hash = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_' + config_hash[:10]
RUN_DIR = ROOT / 'artifacts' / RUN_ID
for directory in ['tables', 'figures', 'models', 'predictions', 'data']:
    (RUN_DIR / directory).mkdir(parents=True, exist_ok=True)
def save_json(name, obj):
    (RUN_DIR / name).write_text(json.dumps(obj, indent=2, default=str, allow_nan=False) + '\n')
def save_table(name, frame):
    frame.to_csv(RUN_DIR / 'tables' / (name + '.csv'), index=False)
def figure(name):
    plt.tight_layout()
    plt.savefig(RUN_DIR / 'figures' / (name + '.png'), dpi=150, bbox_inches='tight')
    display(Image(filename=str(RUN_DIR / 'figures' / (name + '.png'))))
    plt.close()
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
save_json('run_config.json', CONFIG)
if REVIEW_MODE:
    save_json('comparison_source.json', {'run_id':REVIEW_DIR.name,'source_manifest_sha256':sha256(REVIEW_DIR/'manifest.json'),
        'mode':'saved-prediction review; no transformer training or inference',
        'recorded_environment':json.loads((REVIEW_DIR/'environment.json').read_text())})
save_json('source_manifest.json', DATA_MANIFEST)
if IN_COLAB:
    (ROOT/'configs').mkdir(exist_ok=True)
    (ROOT/'configs/run_config.json').write_text(json.dumps({k:v for k,v in CONFIG.items() if k!='execution_mode'},indent=2)+'\n')
    (ROOT/'data').mkdir(exist_ok=True)
    (ROOT/'data/manifest.json').write_text(json.dumps(DATA_MANIFEST,indent=2)+'\n')
(RUN_DIR / 'requirements.txt').write_text(CORE_REQUIREMENTS)
(RUN_DIR / 'requirements-transformer.txt').write_text(GPU_REQUIREMENTS)
packages = sorted(f'{d.metadata["Name"]}=={d.version}' for d in importlib.metadata.distributions() if d.metadata['Name'])
(RUN_DIR / 'environment.freeze.txt').write_text('\n'.join(packages) + '\n')
ENVIRONMENT = {'python':sys.version, 'platform':platform.platform(), 'processor':platform.processor(),
               'colab':IN_COLAB, 'config_sha256':config_hash, 'run_id':RUN_ID,
               'thread_settings':{k:os.environ.get(k) for k in ['OMP_NUM_THREADS','MKL_NUM_THREADS','PYTHONHASHSEED']}}
save_json('environment.json', ENVIRONMENT)
# Save exactly the cells used, without outputs, for source-level reproduction.
notebook_path = ROOT / 'NLP_Simone_Rossi.ipynb'
if notebook_path.exists():
    source_notebook = json.loads(notebook_path.read_text())
    for cell in source_notebook['cells']:
        if cell['cell_type'] == 'code':
            cell['outputs'], cell['execution_count'] = [], None
    save_json('notebook_source.ipynb', source_notebook)
print('Run:', RUN_ID, '| mode:', MODE)

## 3. Data, quality checks, and controlled splits
[ClimateSentiment source card](https://huggingface.co/datasets/climatebert/climate_sentiment) describes expert-labelled paragraphs from corporate reports. The exact revision and SHA-256 checksums are saved in the manifest. Public data is cached under `data/`; a hash mismatch fails rather than silently downloading a different version.

**Protocol:** use the official 1,000 training paragraphs for an 800/200 stratified train/validation split; keep the official 320 test paragraphs for the selected classical model and prespecified transformer seeds. Vocabulary, preprocessing decisions, model selection and routing thresholds use development data only. Neither family is refit on validation.

In [ ]:
from urllib.request import urlretrieve
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
for filename, info in DATA_MANIFEST['files'].items():
    path = DATA_DIR / filename
    if not path.exists():
        url = f"https://huggingface.co/datasets/{DATA_MANIFEST['dataset']}/resolve/{DATA_MANIFEST['revision']}/data/{filename}"
        urlretrieve(url, path)
    assert sha256(path) == info['sha256'], f'Checksum mismatch: {filename}'
    shutil.copy2(path, RUN_DIR / 'data' / filename)

def minimal_clean(text):
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', text)).strip()
def normalize_key(text):
    return minimal_clean(text).casefold()
source_train = pd.read_parquet(next(DATA_DIR.glob('train-*.parquet')))
source_test = pd.read_parquet(next(DATA_DIR.glob('test-*.parquet')))
assert len(source_train) == 1000 and len(source_test) == 320
for frame, prefix in [(source_train, 'train'), (source_test, 'test')]:
    assert frame[['text', 'label']].notna().all().all()
    assert frame['label'].isin(LABELS).all()
    assert frame['text'].map(lambda x:isinstance(x, str) and bool(x.strip())).all()
    frame['row_id'] = [f'{prefix}:{i:04d}' for i in range(len(frame))]
    frame['clean_text'] = frame['text'].map(minimal_clean)
    frame['text_key'] = frame['text'].map(normalize_key)
    frame['word_count'] = frame['clean_text'].str.split().str.len()
    assert frame.groupby('text_key')['label'].nunique().max() == 1, 'Conflicting duplicate labels: review before training.'
assert not set(source_train.text_key) & set(source_test.text_key), 'Exact train/test leakage: review protocol.'
duplicate_count = int(source_train.duplicated('text_key').sum())
development = source_train.drop_duplicates('text_key').copy()
train_df, val_df = train_test_split(development, test_size=CONFIG['validation_fraction'],
                                  stratify=development.label, random_state=SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)
test_df = source_test.reset_index(drop=True).copy()
assert not set(train_df.text_key) & set(val_df.text_key)
assert not set(train_df.row_id) & set(val_df.row_id)
SPLITS = {'train':train_df, 'validation':val_df, 'test':test_df}
split_manifest = pd.concat([f[['row_id','label','text_key']].assign(split=s) for s,f in SPLITS.items()])
split_manifest.to_csv(RUN_DIR / 'split_manifest.csv', index=False)
quality = {'missing_values':0, 'conflicting_duplicates':0, 'train_duplicates_removed':duplicate_count,
           'cross_split_exact_duplicates':0, 'split_sizes':{s:len(f) for s,f in SPLITS.items()}}
save_json('data_quality.json', quality)
display(pd.DataFrame([{'split':s, 'rows':len(f), **{NAMES[c]:int((f.label==c).sum()) for c in LABELS}}
                      for s,f in SPLITS.items()]))
train_share=float((train_df.label==2).mean())
test_share=float((test_df.label==2).mean())
display(Markdown(f"**Population difference:** opportunity paragraphs account for **{train_share:.1%}** of training versus **{test_share:.1%}** of test. That change in label mix is one reason accuracy alone can mislead; macro F1 and per-class recall remain the decision measures. It is not evidence of a temporal trend."))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
counts = pd.DataFrame({s:f.label.value_counts().reindex(LABELS, fill_value=0).to_numpy()
                       for s,f in SPLITS.items()}, index=NAMES)
counts.plot.bar(ax=axes[0], rot=0, color=['#1b6a96','#dd8a33','#697582'])
axes[0].set(title='Label counts by split', ylabel='Paragraphs', xlabel='Source label')
for s,f in SPLITS.items():
    axes[1].hist(f.word_count, bins=np.arange(0, 701, 25), alpha=.45, label=f'{s} (n={len(f)})')
axes[1].set(title='Paragraph lengths', xlabel='Whitespace-separated words', ylabel='Paragraphs')
axes[1].legend()
figure('data_profile')
imbalance = train_df.label.value_counts()
print(f'Train largest/smallest class ratio: {imbalance.max()/imbalance.min():.2f}')
display(train_df.sample(5, random_state=SEED)[['row_id','label','word_count','text']])

### 3.1 Linguistic profiling
Minimal cleaning preserves numbers, negation, punctuation and casing. spaCy's general-English POS tags can describe formal language but do not validate climate semantics. Set `NLP_RUN_POS=true` and install `requirements-pos.txt` to regenerate token-level evidence on **training rows only**. It has no effect on model selection and is excluded from the lightweight run.

In [ ]:
if CONFIG['run_pos']:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    tokens = [{'row_id':rid,'token':t.text,'lemma':t.lemma_,'pos':t.pos_}
              for rid,doc in zip(train_df.row_id, nlp.pipe(train_df.clean_text)) for t in doc]
    tokens_df = pd.DataFrame(tokens)
    save_table('pos_tokens', tokens_df)
    tokens_df.pos.value_counts().plot.bar(color='#1b6a96')
    plt.title('Training-corpus POS profile — general-English tagger')
    plt.ylabel('Tokens')
    figure('pos_profile')
    display(tokens_df.head(10))
else:
    print('Optional POS profiling skipped; enable with NLP_RUN_POS=true.')

## 4. Preprocessing and model alternatives — validation only
We test both **preserving function words** and **English stop-word removal**, holding all other TF-IDF+LR settings fixed. Stop-word lists can remove negation; the cell prints exactly which negation words the installed list removes. We compare vocabulary size, retained feature occurrences, validation macro F1 and paired prediction changes. This tests a plausible alternative without claiming that a score difference isolates negation alone.

A full 2×2×2 comparison covers BoW/TF-IDF, logistic regression/Multinomial Naive Bayes, and unigrams/unigrams+bigrams, with both preprocessing options. LR's `C` is chosen within the same validation split. NB uses fixed α=1. All runs use the same training rows, a 1,000-feature cap, and min_df=5. The cap controls cost but can displace rare jargon when bigrams are added. Selection across many candidates can overfit the 200-row validation set; the test score and uncertainty assess the frozen winner.

In [ ]:
if REVIEW_MODE:
    B=CONFIG['baseline']
    experiments=pd.read_csv(REVIEW_DIR/'tables/validation_experiments.csv')
    SELECTED_NAME=json.loads((REVIEW_DIR/'selected_classical.json').read_text())['model']
    selected_model=joblib.load(REVIEW_DIR/'models/classical.joblib')
    assert selected_model.classes_.tolist()==LABELS
    selected_val_probs=selected_model.predict_proba(val_df.clean_text)
    models={SELECTED_NAME:selected_model}
    validation_predictions={SELECTED_NAME:selected_model.predict(val_df.clean_text)}
    joblib.dump(selected_model,RUN_DIR/'models/classical.joblib')
    save_table('validation_experiments',experiments)
    save_json('selected_classical.json',json.loads((REVIEW_DIR/'selected_classical.json').read_text()))
    print('Selected training-fitted classical pipeline:',SELECTED_NAME)
    print('Review uses the recorded validation search; it does not fit new candidates.')
    display(experiments.head(8).round(4))
else:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    print('Negation words removed by the English list:', sorted({'not','no','nor','never'} & ENGLISH_STOP_WORDS))
    B = CONFIG['baseline']
    models, validation_predictions, experiment_rows = {}, {}, []
    for representation, vectorizer in [('BoW',CountVectorizer),('TF-IDF',TfidfVectorizer)]:
        for ngram in [(1,1),(1,2)]:
            for stop_words in [None, 'english']:
                for classifier in ['LR','NB']:
                    for C in (B['C_grid'] if classifier=='LR' else [None]):
                        name = f'{representation}|{classifier}|{ngram[1]}gram|stop={stop_words}|C={C}'
                        clf = (LogisticRegression(C=C, max_iter=B['max_iter'], random_state=SEED)
                               if classifier=='LR' else MultinomialNB(alpha=B['alpha']))
                        pipeline = Pipeline([('vectorizer',vectorizer(stop_words=stop_words,
                              ngram_range=ngram, min_df=B['min_df'], max_features=B['max_features'])),('classifier',clf)])
                        start = time.perf_counter()
                        pipeline.fit(train_df.clean_text, train_df.label)
                        fit_seconds = time.perf_counter()-start
                        prediction = pipeline.predict(val_df.clean_text)
                        models[name] = pipeline
                        validation_predictions[name] = prediction
                        experiment_rows.append({'model':name,'representation':representation,'classifier':classifier,
                            'ngram_max':ngram[1], 'stop_words':str(stop_words),'C':C,
                            'validation_macro_f1':f1_score(val_df.label,prediction,labels=LABELS,average='macro',zero_division=0),
                            'validation_risk_recall':recall_score(val_df.label,prediction,labels=[0],average='macro',zero_division=0),
                            'fit_seconds':fit_seconds,'vocabulary_size':len(pipeline[0].vocabulary_)})
    experiments = pd.DataFrame(experiment_rows).sort_values(['validation_macro_f1','model'],ascending=[False,True]).reset_index(drop=True)
    save_table('validation_experiments', experiments)
    SELECTED_NAME = experiments.iloc[0]['model']
    selected_model = models[SELECTED_NAME]
    selected_val_probs = selected_model.predict_proba(val_df.clean_text)
    assert selected_model.classes_.tolist() == LABELS
    joblib.dump(selected_model, RUN_DIR/'models/classical.joblib')
    save_json('selected_classical.json', experiments.iloc[0].to_dict())
    print('Frozen classical choice:', SELECTED_NAME)
    display(experiments.head(8).round(4))

In [ ]:
if REVIEW_MODE:
    preprocessing=pd.read_csv(REVIEW_DIR/'tables/preprocessing_comparison.csv')
    save_table('preprocessing_comparison',preprocessing)
    display(preprocessing.round(4))
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    axes[0].bar(preprocessing.preprocessing,preprocessing.mean_retained_feature_occurrences,color=['#1b6a96','#dd8a33'])
    axes[0].set(title='Retained lexical evidence — recorded training experiment',ylabel='Mean feature occurrences / paragraph')
    axes[1].bar(preprocessing.preprocessing,preprocessing.validation_macro_f1,color=['#1b6a96','#dd8a33'])
    axes[1].set(title='Controlled stop-word comparison (validation n=200)',ylabel='Macro F1',ylim=(0,1))
    figure('preprocessing_comparison')
    delta=float(preprocessing.iloc[1].validation_macro_f1-preprocessing.iloc[0].validation_macro_f1)
    display(Markdown(f'**Measured preprocessing effect:** stop-word removal changes validation macro F1 by {delta:+.4f} with the same LR setting. Both vocabularies contain 1,000 features; the occurrence counts show that different lexical evidence survives. This comparison does not isolate negation as the cause. Source: `tables/preprocessing_comparison.csv`.'))
else:
    preprocessing_rows=[]
    for stops in [None, 'english']:
        name=f'TF-IDF|LR|1gram|stop={stops}|C=1.0'
        p=models[name]
        analyzer=p[0].build_analyzer()
        vocab=p[0].vocabulary_
        occurrences=[sum(t in vocab for t in analyzer(x)) for x in train_df.clean_text]
        preprocessing_rows.append({'preprocessing':f'stop_words={stops}',
           'vocabulary_size':len(vocab),'mean_retained_feature_occurrences':np.mean(occurrences),
           'validation_macro_f1':f1_score(val_df.label, validation_predictions[name],labels=LABELS,average='macro',zero_division=0)})
    preprocessing=pd.DataFrame(preprocessing_rows)
    base=preprocessing.iloc[0].validation_macro_f1
    preprocessing['delta_f1_vs_preserve']=preprocessing.validation_macro_f1-base
    save_table('preprocessing_comparison',preprocessing)
    display(preprocessing.round(4))
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    axes[0].bar(preprocessing.preprocessing,preprocessing.mean_retained_feature_occurrences,color=['#1b6a96','#dd8a33'])
    axes[0].set(ylabel='Mean feature occurrences / train paragraph',title='How much lexical evidence is retained?')
    axes[1].bar(preprocessing.preprocessing,preprocessing.validation_macro_f1,color=['#1b6a96','#dd8a33'])
    axes[1].set(ylim=(0,1),ylabel='Validation macro F1',title='Controlled preprocessing comparison (n=200)')
    for i,v in enumerate(preprocessing.validation_macro_f1):axes[1].text(i,v+.015,f'{v:.3f}',ha='center')
    figure('preprocessing_comparison')
    changed = validation_predictions['TF-IDF|LR|1gram|stop=None|C=1.0'] != validation_predictions['TF-IDF|LR|1gram|stop=english|C=1.0']
    print(f'Preprocessing changes {changed.sum()}/{len(val_df)} validation predictions.')
    display(val_df.loc[changed,['row_id','label','text']].head(3))
    change=float(preprocessing.loc[preprocessing.preprocessing=="stop_words=english","delta_f1_vs_preserve"].iloc[0])
    display(Markdown(f"**Preprocessing finding:** removing English stop words changes validation macro F1 by **{change:+.4f}** versus preserving them. Both vocabularies hit the 1,000-feature cap, but the retained occurrence counts differ. The evidence concerns this fixed representation and classifier; it does not isolate negation as the causal explanation."))

In [ ]:
# Best C within each otherwise fixed comparison; every score is validation, not test.
ablation=experiments.sort_values('validation_macro_f1',ascending=False).drop_duplicates(
    ['representation','classifier','ngram_max','stop_words']).copy()
reference=float(experiments.loc[experiments.model=='TF-IDF|LR|1gram|stop=None|C=1.0','validation_macro_f1'].iloc[0])
ablation['delta_vs_reference']=ablation.validation_macro_f1-reference
save_table('validation_ablation',ablation)
ordered=ablation.sort_values('validation_macro_f1')
fig,ax=plt.subplots(figsize=(11,7))
labels=[f'{r.representation} / {r.classifier} / 1–{r.ngram_max} grams / stop={r.stop_words}' for r in ordered.itertuples()]
ax.barh(labels,ordered.validation_macro_f1,color='#1b6a96')
ax.set(xlim=(0,1),xlabel='Validation macro F1',title='Controlled model alternatives — same 800 training / 200 validation rows')
for i,v in enumerate(ordered.validation_macro_f1):ax.text(v+.01,i,f'{v:.3f}',va='center',fontsize=9)
figure('validation_ablation')
# Pure pairs differ only in the named factor; positive delta favors the second value.
pair_rows=[]
for factor,lo,hi in [('representation','BoW','TF-IDF'),('classifier','NB','LR'),('ngram_max',1,2),('stop_words','None','english')]:
    controls=[c for c in ['representation','classifier','ngram_max','stop_words'] if c!=factor]
    for keys,g in ablation.groupby(controls):
        lookup=g.set_index(factor).validation_macro_f1
        if lo in lookup and hi in lookup:
            pair_rows.append({'factor':factor,'controls':str(keys),'comparison':f'{hi} minus {lo}',
                              'delta_validation_f1':float(lookup[hi]-lookup[lo])})
paired_ablations=pd.DataFrame(pair_rows)
save_table('paired_ablation_deltas',paired_ablations)
display(paired_ablations.groupby('factor').delta_validation_f1.agg(['min','median','max']).round(4))
print('Effects depend on the controls; no blanket claim that LR, TF-IDF or bigrams always wins.')

### 4.1 Training vocabulary and overlap
Vocabulary overlap is descriptive and is fitted on training text only. Shared climate terminology may be useful for the domain yet insufficient to distinguish labels. The top-term plot shows class-average TF-IDF, not causal word importance.

In [ ]:
descriptive_vectorizer=TfidfVectorizer(min_df=5,max_features=1000,stop_words=None)
X_train_descriptive=descriptive_vectorizer.fit_transform(train_df.clean_text)
terms=descriptive_vectorizer.get_feature_names_out()
top_sets=[]
fig,axes=plt.subplots(1,3,figsize=(14,4.8))
for label,ax in zip(LABELS,axes):
    mean=np.asarray(X_train_descriptive[train_df.label.to_numpy()==label].mean(axis=0)).ravel()
    indices=mean.argsort()[-10:]
    ax.barh(terms[indices],mean[indices],color=COLORS[label])
    ax.set(title=f'{NAMES[label]} — training terms',xlabel='Mean TF-IDF / paragraph')
    top_sets.append(set(terms[mean.argsort()[-100:]]))
figure('top_training_terms')
overlap=np.array([[len(a&b)/len(a|b) for b in top_sets] for a in top_sets])
fig,ax=plt.subplots(figsize=(5.5,4.7))
image=ax.imshow(overlap,vmin=0,vmax=1,cmap='Blues')
ax.set(xticks=LABELS,yticks=LABELS,xticklabels=NAMES,yticklabels=NAMES,title='Top-100 vocabulary Jaccard overlap')
for i in LABELS:
    for j in LABELS:ax.text(j,i,f'{overlap[i,j]:.2f}',ha='center',va='center',color='white' if overlap[i,j]>.5 else '#202830')
fig.colorbar(image,ax=ax,label='Intersection / union')
figure('vocabulary_overlap')

## 5. Freeze routing before test evaluation
A single threshold is chosen using validation only, with explicit retained accuracy, coverage and support requirements. If no threshold qualifies, **every paragraph goes to review**. This implements a fail-closed policy; it does not establish calibration or fairness. Length-aware thresholds would need enough validation examples per length slice and a new confirmation set.

In [ ]:
def routing_policy(y, probs):
    pred=probs.argmax(axis=1); confidence=probs.max(axis=1)
    rows=[]
    for threshold in CONFIG['routing_threshold_grid']:
        accepted=confidence>=threshold
        rows.append({'threshold':threshold,'retained_n':int(accepted.sum()),'coverage':float(accepted.mean()),
                     'retained_accuracy':float(np.mean(pred[accepted]==np.asarray(y)[accepted])) if accepted.any() else np.nan})
    table=pd.DataFrame(rows)
    D=CONFIG['decision']
    eligible=table[(table.retained_n>=D['min_validation_retained_n']) &
                   (table.coverage>=D['min_coverage']) & (table.retained_accuracy>=D['min_retained_accuracy'])]
    threshold=float(eligible.sort_values(['coverage','threshold'],ascending=[False,True]).iloc[0].threshold) if len(eligible) else None
    return threshold,table
classical_threshold,classical_routing_validation=routing_policy(val_df.label,selected_val_probs)
save_table('classical_routing_validation',classical_routing_validation)
save_json('classical_routing_policy.json',{'threshold':classical_threshold,'source':'validation only','if_null':'review all'})
display(classical_routing_validation.round(3))
print('Frozen routing threshold:',classical_threshold,'(None means review all)')

## 6. Fine-tune DistilBERT
DistilBERT is a smaller contextual alternative to full BERT. We use a pinned base checkpoint, a fresh three-class head, learning rate 2e-5, batch size 16, FP32, weight decay 0.01, and at most three epochs. Dynamic padding reduces wasted work. Early stopping selects the best validation macro F1 and stops after one epoch without improvement. Seeds are set **before head initialization**.

**Constraint trade-off:** max_length=128 limits memory and runtime but may discard a paragraph's conclusion. The code measures actual subword truncation and saves its slice performance. Early stopping is conditional on a validation plateau; the three-epoch cap also bounds compute. Actual per-epoch logs are saved rather than assuming that a plateau or overfitting occurs. A 256-token or chunked model is an untested improvement to evaluate if truncation errors are concentrated.

Full runs train seeds 42/43/44 on the same rows; seed 42 is the prespecified primary comparison, not the best test seed. Secondary seeds quantify initialization/order variation. CPU runs keep transformer fields **missing**, never zero, and do not imply a successful fine-tuning validation.

In [ ]:
HAS_CUDA=False
if MODE not in ('baseline','review'):
    try:
        import torch
        HAS_CUDA=torch.cuda.is_available()
    except ImportError:
        pass
if MODE=='full' and not HAS_CUDA:
    raise RuntimeError('Full profile requires CUDA. Select a Colab GPU kernel, install GPU requirements, restart, then Run all.')
RUN_TRANSFORMER=MODE not in ('baseline','review') and HAS_CUDA
transformer_runs=[]
transformer_status='pending: CUDA run not performed'
if RUN_TRANSFORMER:
    os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
    os.environ['USE_TF']='0'
    from transformers import (AutoTokenizer,AutoModelForSequenceClassification,DataCollatorWithPadding,
                              Trainer,TrainingArguments,EarlyStoppingCallback,set_seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.backends.cuda.matmul.allow_tf32=False
    tokenizer=AutoTokenizer.from_pretrained(CONFIG['model_id'],revision=CONFIG['model_revision'])
    T=CONFIG['transformer']
    token_lengths={s:np.array([len(ids) for ids in tokenizer(f.clean_text.tolist(),truncation=False)['input_ids']]) for s,f in SPLITS.items()}
    truncation=pd.DataFrame([{'split':s,'n':len(lengths),'truncated_n':int((lengths>T['max_length']).sum()),
                              'truncated_fraction':float((lengths>T['max_length']).mean()),'p95_subwords':float(np.quantile(lengths,.95))}
                             for s,lengths in token_lengths.items()])
    save_table('subword_truncation',truncation)
    display(truncation.round(3))
    # Only input_ids and attention_mask are produced by the pinned DistilBERT tokenizer.
    class ParagraphDataset(torch.utils.data.Dataset):
        def __init__(self, frame):
            self.encodings=tokenizer(frame.clean_text.tolist(),truncation=True,max_length=T['max_length'])
            self.labels=frame.label.tolist()
        def __len__(self):return len(self.labels)
        def __getitem__(self, index):
            return {**{key:values[index] for key,values in self.encodings.items()},'label':self.labels[index]}
    tokenized={s:ParagraphDataset(f) for s,f in SPLITS.items()}
    def hf_metrics(output):
        pred=output.predictions.argmax(axis=1)
        return {'macro_f1':f1_score(output.label_ids,pred,labels=LABELS,average='macro',zero_division=0),
                'accuracy':accuracy_score(output.label_ids,pred)}
    ENVIRONMENT.update({'torch':torch.__version__,'cuda_version':torch.version.cuda,
                        'gpu':torch.cuda.get_device_name(0),'deterministic_algorithms':True,'fp16':False})
    save_json('environment.json',ENVIRONMENT)
elif not REVIEW_MODE:
    print(transformer_status+'. Classical analysis will complete; transformer-dependent gates remain pending.')
if REVIEW_MODE:
    recorded_resources=pd.read_csv(REVIEW_DIR/'tables/resource_comparison.csv').set_index('model')
    for run_seed in CONFIG['transformer']['seeds']:
        name=f'DistilBERT seed {run_seed}'
        saved=pd.read_csv(REVIEW_DIR/'predictions'/f'distilbert_seed_{run_seed}.csv')
        assert saved.row_id.tolist()==test_df.row_id.tolist()
        checkpoint=REVIEW_DIR/'models'/f'distilbert_seed_{run_seed}'
        history=json.loads((checkpoint/'trainer_state.json').read_text())['log_history']
        resource=recorded_resources.loc[name]
        threshold=json.loads((checkpoint/'routing_policy.json').read_text())['threshold']
        save_table(f'transformer_seed_{run_seed}_routing_validation',pd.read_csv(REVIEW_DIR/'tables'/f'transformer_seed_{run_seed}_routing_validation.csv'))
        transformer_runs.append({'seed':run_seed,'probs':saved[['p_risk','p_neutral','p_opportunity']].to_numpy(),
            'threshold':threshold,'history':history,
            **{k:float(resource[k]) for k in ['fit_seconds','inference_ms_per_paragraph','artifact_mb','peak_gpu_mb']}})
        (RUN_DIR/'models'/f'distilbert_seed_{run_seed}').mkdir(exist_ok=True)
        save_json(f'models/distilbert_seed_{run_seed}/trainer_state.json',json.loads((checkpoint/'trainer_state.json').read_text()))
        for filename in ['training_arguments.json','routing_policy.json']:
            save_json(f'models/distilbert_seed_{run_seed}/{filename}',json.loads((checkpoint/filename).read_text()))
    truncation=pd.read_csv(REVIEW_DIR/'tables/subword_truncation.csv')
    save_table('subword_truncation',truncation)
    display(truncation.round(3))
    transformer_status='complete recorded GPU evidence; reviewed from saved predictions, not retrained'
    print(transformer_status)

In [ ]:
if RUN_TRANSFORMER:
    for run_seed in CONFIG['transformer']['seeds']:
        set_seed(run_seed)
        model=AutoModelForSequenceClassification.from_pretrained(CONFIG['model_id'],revision=CONFIG['model_revision'],
                num_labels=3,id2label=dict(enumerate(NAMES)),label2id={name:i for i,name in enumerate(NAMES)},
                attn_implementation='eager')
        checkpoint_dir=RUN_DIR/'models'/f'distilbert_seed_{run_seed}'
        args=TrainingArguments(output_dir=str(checkpoint_dir/'checkpoints'),num_train_epochs=T['epochs'],
             learning_rate=T['learning_rate'],per_device_train_batch_size=T['batch_size'],
             per_device_eval_batch_size=T['batch_size'],weight_decay=T['weight_decay'],warmup_ratio=T['warmup_ratio'],
             eval_strategy='epoch',save_strategy='epoch',logging_strategy='steps',logging_steps=10,
             load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,
             save_total_limit=1,seed=run_seed,data_seed=run_seed,full_determinism=True,fp16=False,bf16=False,
             optim='adamw_torch',report_to='none',dataloader_num_workers=0,dataloader_pin_memory=True)
        trainer=Trainer(model=model,args=args,train_dataset=tokenized['train'],eval_dataset=tokenized['validation'],
              processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=hf_metrics,
              callbacks=[EarlyStoppingCallback(early_stopping_patience=T['early_stopping_patience'])])
        torch.cuda.reset_peak_memory_stats()
        start=time.perf_counter()
        trainer.train()
        torch.cuda.synchronize()
        train_seconds=time.perf_counter()-start
        trainer.save_model(str(checkpoint_dir/'best'))
        tokenizer.save_pretrained(checkpoint_dir/'best')
        trainer.state.save_to_json(str(checkpoint_dir/'trainer_state.json'))
        save_json(f'models/distilbert_seed_{run_seed}/training_arguments.json',args.to_dict())
        val_logits=trainer.predict(tokenized['validation']).predictions
        val_probs=torch.softmax(torch.tensor(val_logits),dim=-1).numpy()
        threshold,routing_table=routing_policy(val_df.label,val_probs)
        save_table(f'transformer_seed_{run_seed}_routing_validation',routing_table)
        # Warm-up is excluded from timed prediction. Test labels are not used for selection.
        trainer.predict(torch.utils.data.Subset(tokenized['test'],range(min(16,len(test_df)))))
        torch.cuda.synchronize()
        start=time.perf_counter()
        output=trainer.predict(tokenized['test'])
        torch.cuda.synchronize()
        inference_seconds=time.perf_counter()-start
        test_probs=torch.softmax(torch.tensor(output.predictions),dim=-1).numpy()
        transformer_runs.append({'seed':run_seed,'probs':test_probs,'val_probs':val_probs,'threshold':threshold,
            'fit_seconds':train_seconds,'inference_ms_per_paragraph':1000*inference_seconds/len(test_df),
            'artifact_mb':sum(p.stat().st_size for p in (checkpoint_dir/'best').rglob('*') if p.is_file())/1e6,
            'peak_gpu_mb':torch.cuda.max_memory_allocated()/1e6,'history':trainer.state.log_history})
        np.save(checkpoint_dir/'test_probabilities.npy',test_probs)
        np.save(checkpoint_dir/'validation_probabilities.npy',val_probs)
        save_json(f'models/distilbert_seed_{run_seed}/routing_policy.json',{'threshold':threshold,'source':'validation'})
        del trainer,model
        torch.cuda.empty_cache()
    transformer_status='complete: controlled GPU training and predictions saved'
    print(transformer_status)

In [ ]:
if transformer_runs:
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    for result in transformer_runs:
        evaluation=[r for r in result['history'] if 'eval_macro_f1' in r]
        axes[0].plot([r['epoch'] for r in evaluation],[r['eval_loss'] for r in evaluation],marker='o',label=f"seed {result['seed']}")
        axes[1].plot([r['epoch'] for r in evaluation],[r['eval_macro_f1'] for r in evaluation],marker='o',label=f"seed {result['seed']}")
    axes[0].set(title='Recorded validation loss',xlabel='Epoch',ylabel='Cross-entropy loss')
    axes[1].set(title='Recorded validation macro F1',xlabel='Epoch',ylabel='Macro F1',ylim=(0,1))
    for ax in axes:ax.legend()
    figure('controlled_training_curves')
else:
    print('Training curves unavailable: this execution contains no transformer evidence.')

## 7. Locked-model test evaluation and uncertainty
Only the selected classical candidate and a majority-class sanity baseline are evaluated here; unselected candidates remain validation-only. Transformer seed 42 uses the same test row order when present. Stratified paired bootstrap resamples the same test indices for both models, preserving the observed class counts. Its 95% interval is conditional on this sample and fitted models; it cannot establish issuer-level independence or deployment readiness.

In [ ]:
def metrics(y,probs):
    pred=probs.argmax(axis=1)
    return {'accuracy':accuracy_score(y,pred),
            'macro_f1':f1_score(y,pred,labels=LABELS,average='macro',zero_division=0),
            'micro_f1':f1_score(y,pred,labels=LABELS,average='micro',zero_division=0),
            'macro_average_precision':average_precision_score(label_binarize(y,classes=LABELS),probs,average='macro'),
            'risk_recall':recall_score(y,pred,labels=[0],average='macro',zero_division=0)}
selected_model.predict_proba(test_df.clean_text.iloc[:16])  # warm-up
start=time.perf_counter()
classical_probs=selected_model.predict_proba(test_df.clean_text)
classical_ms=1000*(time.perf_counter()-start)/len(test_df)
majority=DummyClassifier(strategy='most_frequent').fit(train_df.clean_text,train_df.label)
majority_probs=majority.predict_proba(test_df.clean_text)
MODEL_PROBS={'Majority baseline':majority_probs,'Selected classical':classical_probs}
for result in transformer_runs:MODEL_PROBS[f"DistilBERT seed {result['seed']}"]=result['probs']
y_true=test_df.label.to_numpy()
rng=np.random.default_rng(SEED)
bootstrap_indices=np.array([np.concatenate([rng.choice(np.flatnonzero(y_true==c),size=int((y_true==c).sum()),replace=True)
                                           for c in LABELS]) for _ in range(CONFIG['bootstrap_resamples'])])
def bootstrap_f1(pred):
    # Confusion-matrix arithmetic avoids repeatedly fitting or invoking model predictions.
    draws=[]
    for idx in bootstrap_indices:
        cm=np.bincount(3*y_true[idx]+pred[idx],minlength=9).reshape(3,3)
        denominator=cm.sum(0)+cm.sum(1)
        draws.append(np.divide(2*cm.diagonal(),denominator,out=np.zeros(3,dtype=float),where=denominator!=0).mean())
    return np.array(draws)
metric_rows=[]; bootstrap_scores={}
for name,probs in MODEL_PROBS.items():
    assert probs.shape==(len(test_df),3) and np.isfinite(probs).all()
    assert np.allclose(probs.sum(1),1,atol=1e-5)
    stats=metrics(y_true,probs)
    bootstrap_scores[name]=bootstrap_f1(probs.argmax(1))
    low,high=np.quantile(bootstrap_scores[name],[.025,.975])
    metric_rows.append({'model':name,**stats,'macro_f1_ci_low':low,'macro_f1_ci_high':high,'test_n':len(test_df)})
    predictions=test_df[['row_id','text','label','word_count']].copy()
    predictions['prediction']=probs.argmax(1)
    predictions['confidence']=probs.max(1)
    for c in LABELS:predictions[f'p_{NAMES[c]}']=probs[:,c]
    predictions.to_csv(RUN_DIR/'predictions'/(re.sub(r'[^a-z0-9]+','_',name.lower())+'.csv'),index=False)
    save_table(re.sub(r'[^a-z0-9]+','_',name.lower())+'_class_metrics',
               pd.DataFrame(classification_report(y_true,probs.argmax(1),labels=LABELS,target_names=NAMES,output_dict=True,zero_division=0)).T.reset_index(names='class'))
metric_table=pd.DataFrame(metric_rows)
save_table('test_metrics',metric_table)
display(metric_table.round(4))
if REVIEW_MODE:
    recorded=pd.read_csv(REVIEW_DIR/'tables/test_metrics.csv').set_index('model').sort_index()
    recomputed=metric_table.set_index('model').sort_index()
    for metric in ['accuracy','macro_f1','micro_f1','macro_average_precision','risk_recall','macro_f1_ci_low','macro_f1_ci_high']:
        assert np.allclose(recorded[metric],recomputed[metric],atol=1e-10),f'Review mismatch: {metric}'
    print('All saved comparison metrics and bootstrap bounds reproduced from the retained predictions.')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
y=np.arange(len(metric_table))
axes[0].errorbar(metric_table.macro_f1,y,
     xerr=[metric_table.macro_f1-metric_table.macro_f1_ci_low,metric_table.macro_f1_ci_high-metric_table.macro_f1],
     fmt='o',capsize=4,color='#1b6a96')
axes[0].set(yticks=y,yticklabels=metric_table.model,xlim=(0,1),xlabel='Macro F1; 95% stratified bootstrap interval',
            title=f'Official test comparison (n={len(test_df)})')
axes[0].axvline(CONFIG['decision']['min_macro_f1'],ls='--',color='#8b572a',label='Pilot target')
axes[0].legend(loc='lower right')
axes[1].bar(metric_table.model,metric_table.risk_recall,color='#b54736')
axes[1].axhline(CONFIG['decision']['min_risk_recall'],ls='--',color='#8b572a',label='Risk recall target')
axes[1].set(ylim=(0,1.04),ylabel='Recall of labelled risk paragraphs',title='How many risk paragraphs are found?')
axes[1].tick_params(axis='x',rotation=30)
axes[1].legend()
figure('test_comparison')
paired_result=None
if transformer_runs:
    delta_samples=bootstrap_scores['DistilBERT seed 42']-bootstrap_scores['Selected classical']
    delta=float(metric_table.set_index('model').loc['DistilBERT seed 42','macro_f1']-metric_table.set_index('model').loc['Selected classical','macro_f1'])
    low,high=np.quantile(delta_samples,[.025,.975])
    paired_result={'primary_seed':42,'delta_macro_f1':delta,'ci_low':float(low),'ci_high':float(high),
                   'method':'paired class-stratified percentile bootstrap','resamples':CONFIG['bootstrap_resamples']}
    save_json('paired_comparison.json',paired_result)
    print(f'Primary transformer minus classical: {delta:+.4f} F1; 95% paired CI [{low:+.4f}, {high:+.4f}]')
    secondary=metric_table[metric_table.model.str.startswith('DistilBERT')]
    print(f'Prespecified-seed test F1 mean={secondary.macro_f1.mean():.4f}, SD={secondary.macro_f1.std(ddof=1):.4f}; 3 seeds is limited evidence.')
else:
    print('Transformer comparison unavailable in this execution; no transformer result is inferred.')

### 7.2 Why the balanced score misses 0.80
For each category we ask two questions. **Precision:** when the model chooses this category, how often does that agree with the dataset? **Recall:** among all paragraphs belonging to this category, how many did it find? F1 combines those answers using a harmonic mean, so strong performance on one cannot completely hide weakness on the other. Macro F1 averages the three category F1 scores equally, even though there are 163 neutral and only 51 opportunity paragraphs in this test. The calculation below is independently reconstructed from predictions, using the [F1 definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html).

In [ ]:
class_rows=[]
for name in ['Selected classical','DistilBERT seed 42']:
    if name not in MODEL_PROBS:continue
    cm=confusion_matrix(y_true,MODEL_PROBS[name].argmax(1),labels=LABELS)
    for c in LABELS:
        tp=int(cm[c,c]); predicted_n=int(cm[:,c].sum()); actual_n=int(cm[c,:].sum())
        precision=tp/predicted_n if predicted_n else 0
        recall=tp/actual_n if actual_n else 0
        f1=2*tp/(actual_n+predicted_n) if actual_n+predicted_n else 0
        class_rows.append({'model':name,'class':NAMES[c],'correct':tp,'predicted_n':predicted_n,
             'source_n':actual_n,'missed':actual_n-tp,'false_alarms':predicted_n-tp,
             'precision':precision,'recall':recall,'f1':f1})
class_evidence=pd.DataFrame(class_rows)
save_table('class_evidence',class_evidence)
display(class_evidence.round(4))
fig,ax=plt.subplots(figsize=(10,4.8))
x=np.arange(3)
for k,name in enumerate(class_evidence.model.unique()):
    d=class_evidence[class_evidence.model==name].set_index('class').loc[NAMES]
    ax.bar(x+(k-.5)*.34,d.f1,width=.34,label=name,color=['#1b6a96','#dd8a33'][k])
ax.axhline(CONFIG['decision']['min_macro_f1'],ls='--',color='#7a4822',label='Macro target, shown as equal-class reference')
ax.set(xticks=x,xticklabels=NAMES,ylim=(0,1.05),ylabel='Per-class F1',title='Where balanced quality is lost — official test, n=320')
ax.legend(fontsize=9)
figure('f1_shortfall')
if 'DistilBERT seed 42' in MODEL_PROBS:
    d=class_evidence[class_evidence.model=='DistilBERT seed 42'].set_index('class')
    observed=float(d.f1.mean()); target=CONFIG['decision']['min_macro_f1']
    required_opportunity=3*target-d.loc['risk','f1']-d.loc['neutral','f1']
    assert np.isclose(observed,metric_table.set_index('model').loc['DistilBERT seed 42','macro_f1'])
    note=f"""**Measured reason for missing the target:** the primary transformer's class scores are risk **{d.loc['risk','f1']:.3f}**, neutral **{d.loc['neutral','f1']:.3f}**, and opportunity **{d.loc['opportunity','f1']:.3f}**. Their arithmetic mean is **{observed:.3f}**, leaving **{target-observed:.3f}** to the proposed {target:.2f} macro target. Opportunity is the largest class-level weakness: it finds **{int(d.loc['opportunity','correct'])}/{int(d.loc['opportunity','source_n'])}**, while **{int(d.loc['opportunity','false_alarms'])}/{int(d.loc['opportunity','predicted_n'])}** of its opportunity predictions are false alarms against the dataset. Improving risk recall alone therefore does not satisfy the balanced-quality gate.

If risk and neutral F1 stayed exactly fixed, opportunity F1 would need to reach about **{required_opportunity:.3f}** to make the mean {target:.2f}. This is an arithmetic illustration, not a forecast or an independently achievable intervention: changing predictions also changes other classes' precision and recall. The dashed line in the figure is a reference to the **average** target; the rule does not require every individual class to exceed 0.80.

**What we know about possible causes:** the opportunity→neutral examples in Section 8 show that both specific initiatives and future aspirations can be missed. Training contains 200 opportunity paragraphs versus 326 neutral paragraphs, which could limit coverage of opportunity wording, but no learning-curve experiment isolates that effect. The 128-token limit truncates 57/320 test paragraphs, but their transformer accuracy is 45/57, higher than the overall 243/320; it cannot explain all observed shortfall. The recorded validation F1 of seed 42 reaches 0.842, while test F1 is {observed:.3f}; that difference is consistent with validation-selection optimism or different paragraph composition, and does not by itself establish overfitting or label noise. Sources: `split_manifest.csv`, `tables/error_slices.csv`, seed-42 `trainer_state.json`, and the full error examples.

**Decision implication:** the prespecified model misses the declared project gate in this experiment. It has not demonstrated the proposed level of balanced reliability for unattended use. This is not proof that all its suggestions are wrong, that 0.80 is a universal safety boundary, or that more training would necessarily fix it. A better-targeted data/model experiment and a fresh holdout are needed before changing the recommendation."""
    display(Markdown(note))
    (RUN_DIR/'quality_interpretation.md').write_text(note+'\n')

### 7.1 Resource trade-offs and test routing
Inference is warmed-up batch wall time per paragraph, including the model wrapper; it is **not** single-request p95 latency. CPU and GPU timings describe separate deployment profiles and do not isolate architecture speed. Measure the intended service on its own hardware before adopting it. Fit time excludes classical candidate search; search time is also reported.

In [ ]:
resource_rows=[{'model':'Selected classical','device':'CPU','fit_seconds':float(experiments.iloc[0].fit_seconds),
               'search_fit_seconds':float(experiments.fit_seconds.sum()),'inference_ms_per_paragraph':classical_ms,
               'artifact_mb':(RUN_DIR/'models/classical.joblib').stat().st_size/1e6}]
for r in transformer_runs:
    resource_rows.append({'model':f"DistilBERT seed {r['seed']}",'device':ENVIRONMENT.get('gpu','CUDA'),
            **{k:r[k] for k in ['fit_seconds','inference_ms_per_paragraph','artifact_mb','peak_gpu_mb']}})
resources=pd.read_csv(REVIEW_DIR/'tables/resource_comparison.csv') if REVIEW_MODE else pd.DataFrame(resource_rows)
save_table('resource_comparison',resources)
display(resources.round(3))
policies={'Selected classical':classical_threshold}
policies.update({f"DistilBERT seed {r['seed']}":r['threshold'] for r in transformer_runs})
routing_rows=[]
for name,threshold in policies.items():
    probs=MODEL_PROBS[name]
    accepted=probs.max(1)>=threshold if threshold is not None else np.zeros(len(test_df),dtype=bool)
    routing_rows.append({'model':name,'threshold':threshold,'retained_n':int(accepted.sum()),
        'coverage':float(accepted.mean()),'review_fraction':float(1-accepted.mean()),
        'retained_accuracy':float((probs.argmax(1)[accepted]==y_true[accepted]).mean()) if accepted.any() else np.nan,
        'missed_risk_in_retained':int(((y_true==0)&(probs.argmax(1)!=0)&accepted).sum())})
routing_test=pd.DataFrame(routing_rows)
save_table('routing_test',routing_test)
display(routing_test.round(3))
fig,ax=plt.subplots(figsize=(10,4))
x=np.arange(len(routing_test))
ax.bar(x,routing_test.coverage,label='Retained by frozen policy',color='#167c70')
ax.bar(x,routing_test.review_fraction,bottom=routing_test.coverage,label='Human review',color='#dd8a33')
ax.set(xticks=x,xticklabels=routing_test.model,ylim=(0,1),ylabel='Share of test paragraphs',title='Review workload — thresholds selected on validation only')
ax.legend(loc='upper center',bbox_to_anchor=(.5,-.12),ncol=2)
figure('review_workload')
primary_routing_table=RUN_DIR/'tables/transformer_seed_42_routing_validation.csv'
if primary_routing_table.exists():
    display(Markdown('**Primary transformer: validation threshold evidence.** These are the measured accuracy/coverage trade-offs used for the routing decision. No threshold is selected using test outcomes.'))
    display(pd.read_csv(primary_routing_table).round(4))

## 8. Documented failures and uncertainty
These diagnostics describe test performance; they do not authorize retuning against this test set. Rule-based slices are **proxies**, overlap, and are not demographic fairness groups. Long text means above the **training** 75th percentile. Short text means <15 words, regardless of whether enough such rows exist. The jargon, forward-looking and risk+mitigation patterns are fixed here for investigation; a match does not prove the paragraph's intended sentiment.

In [ ]:
nontrivial=['Selected classical'] + (['DistilBERT seed 42'] if transformer_runs else [])
fig,axes=plt.subplots(1,len(nontrivial),figsize=(5*len(nontrivial),4.5),squeeze=False)
for name,ax in zip(nontrivial,axes.ravel()):
    cm=confusion_matrix(y_true,MODEL_PROBS[name].argmax(1),labels=LABELS)
    im=ax.imshow(cm,cmap='Blues',vmin=0,vmax=max((y_true==c).sum() for c in LABELS))
    ax.set(xticks=LABELS,yticks=LABELS,xticklabels=NAMES,yticklabels=NAMES,xlabel='Predicted',ylabel='Source label',title=name)
    for i in LABELS:
        for j in LABELS:ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>80 else '#202830')
figure('confusion_matrices')
long_cutoff=float(train_df.word_count.quantile(.75))
slice_masks={
 'All paragraphs':np.ones(len(test_df),bool),
 'Short (<15 words)':test_df.word_count.to_numpy()<15,
 'Long (>train Q75)':test_df.word_count.to_numpy()>long_cutoff,
 'Jargon proxy':test_df.clean_text.str.contains(r'\b(?:TCFD|GHG|PRA|FCA)\b|Scope\s+[123]',case=False,regex=True).to_numpy(),
 'Forward-looking proxy':test_df.clean_text.str.contains(r'\b(?:aim|goal|target|plan|commit)\w*\b',case=False).to_numpy(),
 'Risk + mitigation proxy':(test_df.clean_text.str.contains(r'\brisk\w*\b',case=False)&test_df.clean_text.str.contains(r'\b(?:mitigat|reduc|manag)\w*\b',case=False)).to_numpy()
}
if RUN_TRANSFORMER:slice_masks['Truncated at 128 subwords']=token_lengths['test']>CONFIG['transformer']['max_length']
slice_rows=[]
for name in nontrivial:
    pred=MODEL_PROBS[name].argmax(1)
    for slice_name,mask in slice_masks.items():
        n=int(mask.sum())
        slice_rows.append({'model':name,'slice':slice_name,'n':n,'risk_n':int(((y_true==0)&mask).sum()),
          'errors':int(((pred!=y_true)&mask).sum()),'accuracy':accuracy_score(y_true[mask],pred[mask]) if n else np.nan,
          'macro_f1':f1_score(y_true[mask],pred[mask],labels=LABELS,average='macro',zero_division=0) if n else np.nan,
          'risk_recall':float((pred[mask & (y_true==0)]==0).mean()) if (mask & (y_true==0)).any() else np.nan})
slices=pd.DataFrame(slice_rows)
if REVIEW_MODE:
    recorded_slices=pd.read_csv(REVIEW_DIR/'tables/error_slices.csv')
    recorded_truncated=recorded_slices[recorded_slices.slice=='Truncated at 128 subwords']
    slices=pd.concat([slices,recorded_truncated],ignore_index=True)
save_table('error_slices',slices)
display(slices.round(3))
fig,ax=plt.subplots(figsize=(11,5))
for j,name in enumerate(nontrivial):
    d=slices[(slices.model==name)&(slices.n>0)]
    ax.plot(d.slice,d.accuracy,marker='o',linestyle='none',label=name)
ax.set(ylim=(0,1.05),ylabel='Accuracy (overlapping slices)',title=f'Failure probes — long cutoff {long_cutoff:g} training words')
ax.tick_params(axis='x',rotation=25)
ax.legend()
figure('slice_comparison')
print('Three-class macro F1 uses all 3 labels even if a small slice lacks a class; inspect support before comparing slices.')
all_accuracy=float(slices.loc[(slices.model=="Selected classical")&(slices.slice=="All paragraphs"),"accuracy"].iloc[0])
long_accuracy=float(slices.loc[(slices.model=="Selected classical")&(slices.slice=="Long (>train Q75)"),"accuracy"].iloc[0])
short_n=int(slice_masks["Short (<15 words)"].sum())
display(Markdown(f"**Length evidence:** long-paragraph accuracy is **{long_accuracy:.1%}**, versus **{all_accuracy:.1%}** overall; the long slice overlaps the overall population. There are **{short_n}** test rows under 15 words, so this run cannot substantiate a short-text performance gap. Prioritize the actual confusion pairs rather than assuming length causes failure."))

In [ ]:
# Audit a residual near-duplicate risk without fitting vocabulary on test.
from sklearn.metrics.pairwise import cosine_similarity
similarity_vectorizer=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=20000)
train_similarity=similarity_vectorizer.fit_transform(train_df.clean_text)
test_similarity=similarity_vectorizer.transform(test_df.clean_text)
similarities=cosine_similarity(test_similarity,train_similarity)
nearest=similarities.argmax(1)
near=pd.DataFrame({'test_row_id':test_df.row_id,'train_row_id':train_df.row_id.iloc[nearest].to_numpy(),
                   'cosine_similarity':similarities.max(1)})
save_table('nearest_train_neighbors',near)
print(f'Test paragraphs with character TF-IDF similarity ≥0.90 to training: {(near.cosine_similarity>=.90).sum()}/{len(test_df)}')
display(near.sort_values('cosine_similarity',ascending=False).head(5).round(3))
errors=test_df[['row_id','text','label','word_count']].copy()
errors['prediction']=classical_probs.argmax(1)
errors['confidence']=classical_probs.max(1)
errors['failure_pair']=[f'{NAMES[a]} → {NAMES[b]}' for a,b in zip(errors.label,errors.prediction)]
for name,mask in slice_masks.items():errors[name]=mask
errors=errors[errors.label!=errors.prediction].sort_values(['confidence','row_id'],ascending=[False,True])
save_table('classical_errors',errors)
# Up to 20 deterministic, highest-confidence disagreements; not a random error sample.
display(errors[['row_id','label','prediction','confidence','word_count','text']].head(20).round(3))
error_pair=errors.failure_pair.value_counts()
if len(error_pair):
    print('Most frequent documented confusion:',error_pair.index[0],f'(n={error_pair.iloc[0]})')

### 8.1 The five most common primary-transformer error families
The following families rank all nonzero source→prediction confusions for the primary transformer. Counts are exhaustive for the test set; the semantic descriptions are **interpretations of named examples**, not annotations asserting the same cause for every row. “Shared” means both models make that same wrong class prediction. “New” means the classical decision matched the dataset but the transformer decision did not. The classical model also makes one opportunity→risk error; it is retained in the complete prediction CSV.

In [ ]:
family_specs=[
 (2,1,'Opportunity described as neutral','test:0203','test:0023','A 2050 portfolio ambition is missed by both; a paragraph about implemented HVAC energy savings is a transformer-only miss.'),
 (1,0,'Neutral description interpreted as risk','test:0296','test:0060','A statement about what carbon footprints do not measure is treated as risk by both; a neutral risk-management process is a transformer-only error.'),
 (1,2,'Neutral financial/operational reporting interpreted as opportunity','test:0044','test:0002','A fossil-generation asset-sale gain is labelled opportunity by both; a descriptive asset transaction is a transformer-only opportunity prediction.'),
 (0,1,'Risk exposure described as neutral','test:0253','test:0039','Both miss carbon-price exposure within a mitigation plan; the transformer misses operational-cost risk among greener-investment bullets.'),
 (0,2,'Risk outcome interpreted as opportunity','test:0194','test:0266','Both label a cancelled low-carbon facility opportunity; the transformer labels rising electricity-cost exposure opportunity while the classical model calls it neutral.')]
family_rows=[]; qualitative_rows=[]
if 'DistilBERT seed 42' in MODEL_PROBS:
    bp=MODEL_PROBS['Selected classical'].argmax(1);tp=MODEL_PROBS['DistilBERT seed 42'].argmax(1)
    for true,pred,label,r1,r2,interpretation in family_specs:
        b=(y_true==true)&(bp==pred);t=(y_true==true)&(tp==pred)
        family_rows.append({'pattern':label,'source':NAMES[true],'wrong_prediction':NAMES[pred],
           'classical_errors':int(b.sum()),'transformer_errors':int(t.sum()),'shared_same_error':int((b&t).sum()),
           'new_transformer_errors':int(((y_true==true)&(bp==y_true)&(tp==pred)).sum()),
           'classical_errors_corrected':int((b&(tp==y_true)).sum()),'examples':r1+'; '+r2,'example_interpretation':interpretation})
        for rid in [r1,r2]:
            i=test_df.index[test_df.row_id==rid][0]
            qualitative_rows.append({'pattern':label,'row_id':rid,'source_label':NAMES[int(y_true[i])],
               'classical_prediction':NAMES[int(bp[i])],'transformer_prediction':NAMES[int(tp[i])],
               'text':test_df.loc[i,'text'],'interpretation':interpretation})
    families=pd.DataFrame(family_rows).sort_values('transformer_errors',ascending=False)
    assert families.transformer_errors.sum()==int((tp!=y_true).sum())
    save_table('qualitative_error_families',families)
    save_table('qualitative_error_examples',pd.DataFrame(qualitative_rows))
    display(families[['source','wrong_prediction','classical_errors','transformer_errors','shared_same_error','new_transformer_errors']].rename(columns={'source':'True label','wrong_prediction':'Predicted','classical_errors':'Classical errors','transformer_errors':'Transformer errors','shared_same_error':'Shared','new_transformer_errors':'New transformer'}))
    fig,ax=plt.subplots(figsize=(11,6))
    yy=np.arange(len(families))
    ax.barh(yy-.18,families.classical_errors,height=.36,label='Classical',color='#1b6a96')
    ax.barh(yy+.18,families.transformer_errors,height=.36,label='Transformer seed 42',color='#dd8a33')
    ax.set(yticks=yy,yticklabels=[f'{r.source} → {r.wrong_prediction}' for r in families.itertuples()],
           xlabel='Test errors (paragraphs)',title='Five error families ranked by primary-transformer count')
    ax.invert_yaxis();ax.legend()
    figure('qualitative_error_families')
    for f in families.itertuples():
        texts=[r for r in qualitative_rows if r['pattern']==f.pattern]
        detail='\n\n'.join(f"`{r['row_id']}` — source **{r['source_label']}**, classical **{r['classical_prediction']}**, transformer **{r['transformer_prediction']}**:\n\n> {r['text']}" for r in texts)
        display(Markdown(f'**{f.pattern}: {f.transformer_errors} transformer errors; {f.classical_errors} classical errors.** {f.example_interpretation} This is a hypothesis about these examples, not proof of the mechanism across the whole family.\n\n{detail}'))
    transitions=pd.DataFrame({'outcome':np.select([(bp!=y_true)&(tp==y_true),(bp==y_true)&(tp!=y_true),
               (bp!=y_true)&(tp!=y_true)],['corrected','new error','both wrong'],default='both correct')}).value_counts().reset_index(name='n')
    save_table('paired_outcome_counts',transitions)
    display(transitions)
else:
    print('Paired qualitative comparison unavailable without transformer test predictions.')

### Shared patterns, improvements and new weaknesses
The exhaustive family table shows that **opportunity→neutral remains the largest transformer error family (24 cases, 19 shared)**. The transformer corrects ten classical opportunity→neutral cases but introduces four new misses in that direction; it also changes one classical opportunity→risk decision to neutral without making it correct. Therefore a smaller total does not mean the opportunity boundary is solved. The implemented-energy-savings example `test:0023` is a concrete new miss, while `test:0187` (LED/equipment savings) is a classical miss corrected by the transformer. Source: the two prediction CSVs and `tables/qualitative_error_families.csv`.

**Neutral risk/governance wording is another shared boundary problem.** There are 23 transformer neutral→risk errors, including 12 shared with the classical model and 11 newly introduced errors. By contrast, the overlapping risk-plus-mitigation keyword slice improves from 41/61 to 49/61 correct overall. These are compatible findings: improvement on a proxy slice does not imply improvement for every neutral paragraph mentioning risk. `test:0001` is a correction, whereas the near-repeated Daimler descriptions `test:0060`, `test:0121` and `test:0224` are new neutral→risk errors. Repeated wording means those three errors are not three independent demonstrations of a new mechanism. Sources: `tables/error_slices.csv` and the named prediction rows.

**Financial quantities do not reliably tell the model whether the paragraph describes an opportunity.** For example, `test:0044` describes an asset-sale gain but has a neutral source label; `test:0194` mentions a $70 million low-carbon project followed by contract termination and has a risk label. Both models call both opportunity. This illustrates a finance/context hypothesis, rather than proving that amounts cause errors. The local contributions for `test:0194` in Section 9 check which classical features support that decision.

**The transformer also creates 22 errors on paragraphs the classical model gets right:** 11 neutral→risk, four neutral→opportunity, four opportunity→neutral and three risk→neutral. These are model-relative weaknesses, not evidence that each is a general transformer defect. The formatting pair below documents one specific sensitivity; there is no systematic sarcasm assessment. Sources: `tables/paired_error_examples.csv` and `tables/qualitative_error_families.csv`.

**Next experiments, proposed rather than proven improvements:** first, have two domain reviewers label additional training paragraphs distinguishing opportunities, routine sustainability reporting, and mixed risk/mitigation statements. Resolve disagreements before training, and compare per-class precision/recall and all five confusion counts on a fresh issuer/time holdout. This responds directly to the opportunity→neutral and neutral→risk errors. Second, test heading/format variation and 128-versus-256-token or chunked inputs on development data. Check prediction consistency, risk recall, opportunity recall, latency and memory together; longer context is not justified by an assumption that all long paragraphs fail. Sources motivating the experiment: the natural pair below and the 57/320 recorded truncations.

In [ ]:
if 'DistilBERT seed 42' in MODEL_PROBS:
    pair_ids=['test:0035','test:0184']
    plain=test_df.set_index('row_id').loc[pair_ids[0],'text']
    headed=test_df.set_index('row_id').loc[pair_ids[1],'text']
    assert headed.removeprefix('[Decarbonized Society] ')==plain
    records=[]
    for name in ['Selected classical','DistilBERT seed 42']:
        for rid in pair_ids:
            i=test_df.index[test_df.row_id==rid][0];pr=MODEL_PROBS[name][i]
            records.append({'row_id':rid,'model':name,'heading':rid==pair_ids[1],
              'source_label':NAMES[int(y_true[i])],'prediction':NAMES[int(pr.argmax())],
              'p_neutral':float(pr[1]),'p_opportunity':float(pr[2]),
              'top_two_margin':float(np.sort(pr)[-1]-np.sort(pr)[-2])})
    formatting_pair=pd.DataFrame(records)
    save_table('natural_formatting_pair',formatting_pair)
    display(formatting_pair.round(4))
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    for ax,name in zip(axes,['Selected classical','DistilBERT seed 42']):
        d=formatting_pair[formatting_pair.model==name]
        xx=np.arange(2)
        ax.bar(xx-.17,d.p_neutral,width=.34,label='Neutral score',color='#526b80')
        ax.bar(xx+.17,d.p_opportunity,width=.34,label='Opportunity score',color='#167c70')
        ax.set(title=name,xticks=xx,xticklabels=['Body only','Heading + same body'],ylim=(0,1),ylabel='Model score')
        ax.legend(fontsize=9)
    figure('natural_formatting_pair')
    display(Markdown('**Observed formatting sensitivity:** both paragraphs have the opportunity source label, and the assertion verifies that only the heading is added. The transformer flips from opportunity to neutral; its two leading scores differ by only about 0.03 in each case. The classical model remains opportunity. This is a natural consistency probe on two supplied examples, not a systematic robustness rate or a token attribution for the transformer. Their repeated body also illustrates a test-row-dependence threat: the paragraph bootstrap treats rows as independent and may understate uncertainty when similar material repeats.'))

## 9. Explainability: exact linear contributions and word-deletion sensitivity
**Method: other — an exact decomposition of the classical logistic-regression decision, with a word-deletion sensitivity check.** Attention, LIME, SHAP and Integrated Gradients are not used. This choice is auditable because the retained model is linear over TF-IDF features: its feature contributions and intercept must reconstruct its class-score difference exactly. The assertion below verifies that identity on every explained example. [Classifier coefficient definition](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

For an incorrect classical decision, the contrast is the predicted class against the dataset class. For a correct decision, the contrast is the predicted class against its strongest alternative. Positive contributions support the selected class relative to that comparison; negative contributions oppose it. The word-deletion check then removes one word occurrence at a time and measures how the selected class's probability score changes. All word occurrences in each selected paragraph are checked; repeated words keep their character offsets.

**Scope:** these are explanations of the classical classifier. The saved transformer predictions allow row-level comparisons and a natural input-formatting contrast, but the locally absent transformer tensors prevent new transformer token attributions. Accordingly, no conclusion below about a classical coefficient is presented as evidence about transformer attention or its internal reasoning. Example selection is purposeful and listed explicitly; it is not a representative sample of every failure.

In [ ]:
explain_findings={}
explanation_cases=[
    ('test:0061','Correct opportunity: quantified low-carbon investment'),
    ('test:0296','Shared error: negated measurement caveat'),
    ('test:0194','Shared error: investment followed by project cancellation'),
    ('test:0001','Transformer correction: neutral risk-management description'),
    ('test:0035','Special case: decarbonization paragraph without heading'),
    ('test:0184','Special case: identical body with added heading')]
case_rows=[]; feature_rows=[]; deletion_rows=[]
by_id=test_df.set_index('row_id')
if isinstance(selected_model[-1],LogisticRegression):
    vectorizer,linear=selected_model[0],selected_model[-1]
    assert linear.classes_.tolist()==LABELS
    fig,axes=plt.subplots(3,2,figsize=(14,13))
    for ax,(rid,selection_reason) in zip(axes.ravel(),explanation_cases):
        text=by_id.loc[rid,'text']; true=int(by_id.loc[rid,'label'])
        pos=test_df.index[test_df.row_id==rid][0]
        probabilities=MODEL_PROBS['Selected classical'][pos]
        chosen=int(probabilities.argmax())
        contrast=true if chosen!=true else int(np.argsort(probabilities)[-2])
        x=vectorizer.transform([minimal_clean(text)])
        values=x.toarray().ravel()*(linear.coef_[chosen]-linear.coef_[contrast])
        bias=float(linear.intercept_[chosen]-linear.intercept_[contrast])
        margin=float(linear.decision_function(x)[0,chosen]-linear.decision_function(x)[0,contrast])
        assert np.isclose(values.sum()+bias,margin,atol=1e-10)
        active=np.flatnonzero(x.toarray().ravel())
        features=vectorizer.get_feature_names_out()
        for j in active:
            feature_rows.append({'row_id':rid,'term':features[j],'chosen':NAMES[chosen],
                  'contrast':NAMES[contrast],'contribution':float(values[j])})
        shown=active[np.argsort(np.abs(values[active]))[-8:]]
        ax.barh(features[shown],values[shown],color=['#b54736' if v>0 else '#167c70' for v in values[shown]])
        ax.axvline(0,color='#526b80',lw=.8)
        ax.set(title=f'{rid}: {NAMES[chosen]} vs {NAMES[contrast]}',xlabel='Contribution to the classical class-score difference')
        spans=list(re.finditer(r'\b\w+\b',text))
        perturbed=[minimal_clean(text[:m.start()]+' '+text[m.end():]) for m in spans]
        drops=probabilities[chosen]-selected_model.predict_proba(perturbed)[:,chosen]
        for m,drop in zip(spans,drops):
            deletion_rows.append({'row_id':rid,'word':m.group(),'start':m.start(),'end':m.end(),
                                  'drop_in_chosen_class_score':float(drop)})
        supports=sorted([(features[j],float(values[j])) for j in active if values[j]>0],key=lambda v:-v[1])[:3]
        opposes=sorted([(features[j],float(values[j])) for j in active if values[j]<0],key=lambda v:v[1])[:3]
        tr_pred=(int(MODEL_PROBS['DistilBERT seed 42'][pos].argmax()) if 'DistilBERT seed 42' in MODEL_PROBS else None)
        record={'row_id':rid,'selection_reason':selection_reason,'text':text,'source_label':NAMES[true],
             'classical_prediction':NAMES[chosen],'contrast':NAMES[contrast],'classical_score':float(probabilities[chosen]),
             'transformer_prediction':NAMES[tr_pred] if tr_pred is not None else 'unavailable',
             'intercept_difference':bias,'feature_sum':float(values.sum()),'reconstructed_margin':margin,
             'supporting_terms':supports,'opposing_terms':opposes,
             'most_sensitive_word':spans[int(np.argmax(drops))].group(),'largest_deletion_drop':float(max(drops))}
        case_rows.append(record)
    figure('explanation_cases')
    explanations=pd.DataFrame(case_rows)
    save_table('explanation_cases',explanations)
    save_table('explanation_feature_contributions',pd.DataFrame(feature_rows))
    save_table('explanation_word_deletions',pd.DataFrame(deletion_rows))
    save_json('explanation_cases.json',case_rows)
    for c in case_rows:
        support=', '.join(f'`{t}` ({v:+.3f})' for t,v in c['supporting_terms']) or 'no positive active features'
        opposition=', '.join(f'`{t}` ({v:+.3f})' for t,v in c['opposing_terms']) or 'no negative active features'
        display(Markdown(f"**{c['row_id']} — {c['selection_reason']}.** Source: **{c['source_label']}**; classical: **{c['classical_prediction']}**; transformer: **{c['transformer_prediction']}**.\n\n> {c['text']}\n\nThe classical {c['classical_prediction']}-versus-{c['contrast']} margin is supported by {support}; opposing features include {opposition}. Deleting the word occurrence `{c['most_sensitive_word']}` lowers its chosen-class score by {c['largest_deletion_drop']:.3f}. These are measured model associations for this paragraph, not a causal account of the author's intent. Source: `tables/explanation_feature_contributions.csv` and `tables/explanation_word_deletions.csv`."))
    explain_findings={'method':'exact linear contrast plus word deletion','row_ids':[c['row_id'] for c in case_rows],
                      'row_id':'test:0296','scope':'classical LR only; no transformer token attribution'}
    save_json('explainability_finding.json',explain_findings)
else:
    display(Markdown('The selected estimator is not logistic regression; exact LR feature contributions are not applicable in this execution. No LR explanation is claimed.'))
if isinstance(selected_model[-1],LogisticRegression):
    coverage_probe=pd.DataFrame([{'row_id':'test:0194','word':word,'in_selected_vocabulary':word in selected_model[0].vocabulary_}
                                 for word in ['million','project','terminate','permitting','delays']])
    save_table('explanation_vocabulary_probe',coverage_probe)
    display(coverage_probe)

### What the measured token effects suggest
In the correct opportunity example `test:0061`, **`million` contributes +0.838** to the opportunity-versus-neutral class-score difference; deleting an occurrence lowers the opportunity score by about **0.037**. In the cancelled-project error `test:0194`, the same word contributes **+0.329** to opportunity versus risk, and deletion lowers the chosen-class score by **0.075**. The contrasts differ, so these magnitudes are not a like-for-like importance ranking across paragraphs. Together they demonstrate that a numerical finance cue can support both a correct and an incorrect decision. They do not establish that financial amounts alone determine either outcome. Sources: `tables/explanation_feature_contributions.csv` and `tables/explanation_word_deletions.csv`.

The cancellation example makes the **classical feature-coverage failure** concrete. `terminate`, `permitting` and `delays` are absent from the selected unigram vocabulary; deleting each has exactly zero effect on that model's score. In this representation those outcome words cannot directly contribute features, while `million` and `project` can. This motivates a development experiment with a larger training-fitted vocabulary or character/phrase features and reviewed cancellation examples. The transformer also misclassifies this paragraph, so the classical vocabulary limit cannot explain both models' errors, and no gain from expanding features has yet been measured. Sources: `tables/explanation_vocabulary_probe.csv`, the deletion table, and `test:0194` in the paired predictions.

For the shared measurement-caveat error `test:0296`, **`risks` (+0.561)** and **`physical` (+0.262)** support risk over neutral. Even **`not` (+0.164)** contributes in that direction. This demonstrates that retaining a negation word is not equivalent to representing what it negates: the selected unigram representation has no word-order or dependency-scope feature. It supports testing negation/scope-sensitive contrasts on development data; it does not prove that negation is the sole cause of this error. In the neutral risk-management case `test:0001`, `and`, `safety` and `risks` support the classical risk decision, whereas the transformer's recorded decision matches neutral. This links the local explanation to the measured neutral/risk boundary problem without claiming to explain the transformer's internal mechanism. Sources: the same contribution table, `tables/explanation_cases.csv`, and the error-family counts.

For the two decarbonization paragraphs, `energy` supports the classical opportunity decision with and without the heading, and deleting `renewable` lowers that chosen score by about **0.039 / 0.037**. The classical category stays opportunity in both. The transformer changes category across the exact body-plus-heading pair, with a small top-two score margin. Thus the token explanations and the natural formatting contrast illuminate different models and different uncertainties; neither warrants a general claim that one model understands climate intent. Sources: `tables/explanation_feature_contributions.csv`, the deletion table, and `tables/natural_formatting_pair.csv`.


### 9.1 What explanations can and cannot establish
The computed case notes above link actual source labels, predictions and feature scores. A lexical association is evidence about this fitted model's decision, not proof that the word has that meaning in general. In particular, a positive `not` contribution toward risk would not mean that the model understood the scope of negation: a unigram feature records its presence without knowing what it negates. The documented measurement-caveat error is a reason to test scope-sensitive minimal pairs, not to conclude that negation alone caused every error.

Deletion is also limited: removing a word can make a sentence unnatural and changes the normalization of the remaining TF-IDF features. Correlated words, phrases and the choice of comparison class affect the apparent explanation. A high contribution does not establish label correctness, a human rationale, a causal effect on the real world, or demographic fairness. Six purposefully selected paragraphs cannot establish how often a mechanism occurs. Sarcasm has not been systematically annotated or tested here, so no claim is made about sarcasm performance. Opportunity is the least common class in this test, but its weaker results do not prove that training imbalance caused the gap; a learning-curve or controlled data experiment would be needed.

In [ ]:
if transformer_runs:
    trans_pred=MODEL_PROBS['DistilBERT seed 42'].argmax(1)
    base_pred=classical_probs.argmax(1)
    paired_examples=test_df[['row_id','text','label']].copy()
    paired_examples['classical_prediction']=base_pred
    paired_examples['transformer_prediction']=trans_pred
    paired_examples['outcome']=np.select([(base_pred!=y_true)&(trans_pred==y_true),(base_pred==y_true)&(trans_pred!=y_true),
                              (base_pred!=y_true)&(trans_pred!=y_true)],['corrected','new error','both wrong'],default='both correct')
    save_table('paired_error_examples',paired_examples)
    display(paired_examples.groupby('outcome').size().rename('paragraphs').to_frame())
    display(paired_examples[paired_examples.outcome.isin(['corrected','new error'])].groupby('outcome',sort=False).head(3))
else:
    print('Pending paired transformer predictions. No claim that it fixes a particular failure pattern.')

### Evidence standards for qualitative interpretation
The five error families in Section 8 are an exhaustive ranking of the **primary transformer's nonzero source→prediction confusions**, not a claim that every paragraph in a family has the same linguistic cause. Counts cover all 320 test rows. The quoted examples illustrate candidate mechanisms; the machine-readable evidence keeps their full texts and IDs. The separate keyword slices are overlapping probes, not mutually exclusive causes or a demographic fairness assessment. Changes in prediction between models demonstrate changed decisions, not the reason the transformer changed them.

## 10. What the results mean for people using the tool
This section explains the recorded comparison in terms of reading decisions, missed passages and review effort. Facts refer to the generated metric, error and routing tables; proposed safeguards are explicitly recommendations, not measured improvements.

In [ ]:
base=metric_table.set_index('model').loc['Selected classical']
major=metric_table.set_index('model').loc['Majority baseline']
route=routing_test.set_index('model').loc['Selected classical']
if 'DistilBERT seed 42' in MODEL_PROBS:
    t=metric_table.set_index('model').loc['DistilBERT seed 42']
    tpred=MODEL_PROBS['DistilBERT seed 42'].argmax(1)
    correct=int((tpred==y_true).sum())
    opportunity_found=int(((tpred==2)&(y_true==2)).sum())
    opportunity_total=int((y_true==2).sum())
    risk_missed=int(((tpred!=0)&(y_true==0)).sum())
    routing_validation=pd.read_csv(RUN_DIR/'tables/transformer_seed_42_routing_validation.csv')
    r80=routing_validation.set_index('threshold').loc[0.8]
    r90=routing_validation.set_index('threshold').loc[0.9]
    correct80=int(round(r80.retained_n*r80.retained_accuracy))
    correct90=int(round(r90.retained_n*r90.retained_accuracy))
    briefing=f"""Imagine the categories as three trays on an analyst's desk. A useful assistant should put relevant paragraphs in the right tray without filling that tray with unrelated material. The transformer places **{correct} of {len(test_df)} paragraphs** in the dataset's tray. That is better than the classical system, but it leaves **{len(test_df)-correct} disagreements** for a person to resolve. These are matches to expert-provided labels, not checks on whether the company's underlying environmental claims are true. Evidence: `tables/test_metrics.csv` and `tables/paired_error_examples.csv`.

The opportunity tray is the clearest weakness. The transformer finds **{opportunity_found} of {opportunity_total}** paragraphs labelled opportunity. The other **{opportunity_total-opportunity_found}** are put in neutral. An analyst who reads only the opportunity tray could therefore miss almost half of the opportunities represented by this test's labels. It also misses **{risk_missed} risk paragraphs**, so neutral or opportunity must not be treated as permission to disregard a paragraph. The confusion matrices in Section 8 give those counts directly.

The balanced score of **{t.macro_f1:.3f}** gives the three trays equal weight. It is below the proposed **{CONFIG['decision']['min_macro_f1']:.2f}** target because the opportunity tray still has both missed cases and false alarms; Section 7.2 shows the arithmetic. The target is a project decision rule, not a universal boundary between a useful and a useless system. Failing it means that the model has not earned the level of reliance proposed for this pilot. A supervised trial might still test whether suggestions make analysts faster, but no user-time or business-impact experiment has been run.

Raising the model's acceptance threshold does not solve the workload trade-off in the measured validation results. For seed 42, the 0.80 threshold retains **{int(r80.retained_n)}/{len(val_df)} paragraphs**, with **{correct80}/{int(r80.retained_n)} correct** ({r80.retained_accuracy:.2%}), below the proposed 95% target. Raising the threshold to 0.90 gives **{correct90}/{int(r90.retained_n)} correct** in validation but retains only **{r90.coverage:.1%}**, below the proposed 50% coverage target. This finite set of successful validation examples does not guarantee future correctness. Because neither threshold meets all configured requirements, the implemented policy sends every paragraph to review; it does not silently relax the targets. Evidence: `tables/transformer_seed_42_routing_validation.csv` and `tables/routing_test.csv`.

The business choice is therefore conditional. Keep the transformer as the candidate for an analyst-assistance trial because its paired score improves and it corrects some classical errors, while retaining the cheaper classical model as a comparison. Do not authorize unattended filtering, company ratings or environmental claims on this evidence. Reconsider the choice if a separately labelled test set from companies or reporting periods not seen during development clears the quality gates, reviewers can handle the measured queue, and a user study shows useful time savings without more missed risks. These are proposed conditions for further use, not results already demonstrated."""
else:
    briefing=f'This execution contains classical evidence only: macro F1 {base.macro_f1:.3f}. A transformer comparison is unavailable, so it cannot support a model preference.'
display(Markdown(briefing))
(RUN_DIR/'stakeholder_briefing.md').write_text(briefing+'\n')

### 10.1 Fairness gaps observed — and what is unknown
Here “gap” means **unequal measured performance across labels or wording slices**. It does not mean that a protected demographic group has been shown to suffer discrimination. The source supplies paragraph text and a class label, with no validated protected-group, company-group or reporting-date fields. Inferring those groups from names or locations would introduce an additional unvalidated measurement. Accordingly, the report cannot establish demographic, geographic or issuer-level fairness; a deployment needs appropriately governed group information and a separate evaluation. Evidence: the schema check in Section 3 and [the source card](data/SOURCE_README.md).

There is a concrete class-performance gap: the transformer retrieves 91/106 risk paragraphs but only 27/51 opportunity paragraphs. This 32.9-percentage-point recall gap is smaller than the classical model's gap, but opportunity paragraphs still receive less reliable retrieval. That could matter to companies or analysts whose relevant initiatives are omitted if only one predicted category is read. This is a **potential consequence of the observed misses**, not a measured harm to a company. The figure shows uncertainty and support so the smaller opportunity sample is not treated as equally precise evidence.

In [ ]:
def wilson_interval(k,n,z=1.959963984540054):
    if n==0:return (np.nan,np.nan)
    p=k/n;denom=1+z*z/n
    centre=(p+z*z/(2*n))/denom
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/denom
    return centre-half,centre+half
# Explicit column lookup avoids 'class' being renamed by itertuples.
fairness_table=class_evidence[['model','class','correct','source_n','recall']].copy()
intervals=[wilson_interval(int(r['correct']),int(r['source_n'])) for _,r in fairness_table.iterrows()]
fairness_table['recall_ci_low']=[a for a,b in intervals]
fairness_table['recall_ci_high']=[b for a,b in intervals]
save_table('class_recall_gaps',fairness_table)
fig,ax=plt.subplots(figsize=(10,5))
for k,name in enumerate(fairness_table.model.unique()):
    d=fairness_table[fairness_table.model==name].set_index('class').loc[NAMES]
    y=np.arange(3)+(k-.5)*.12
    ax.errorbar(d.recall,y,xerr=[d.recall-d.recall_ci_low,d.recall_ci_high-d.recall],fmt='o',capsize=4,label=name)
ax.set(yticks=np.arange(3),yticklabels=[f'{label} (n={int(class_evidence[class_evidence["class"]==label].iloc[0].source_n)})' for label in NAMES],
       xlabel='Recall; descriptive 95% Wilson intervals',xlim=(0,1.02),title='Observed class retrieval gaps — not a demographic fairness test')
ax.legend();figure('class_recall_gaps')
display(fairness_table.round(3))
display(Markdown('Intervals treat rows as independent Bernoulli outcomes within a class. Shared issuers and near-repeated paragraphs violate that approximation; these are descriptive intervals, not a cluster-adjusted fairness guarantee. No causal explanation for a class gap is inferred from the intervals.'))

Wording slices show why a single overall score is insufficient. On the 44-paragraph jargon proxy, overall transformer accuracy is 35/44 versus 34/44 for the classical model, yet risk recall falls from 13/14 to 12/14. That is a one-paragraph difference within a small risk subset, so it warrants review rather than a claim of established jargon discrimination. The forward-looking proxy has 86/114 correct transformer predictions; risk recall remains 18/24 for both models. These slices overlap and differ in label mix, so their raw score differences do not isolate an effect of jargon or future wording. Evidence: `tables/error_slices.csv`.

### 10.2 Impacted stakeholders
**Analysts and their managers** could receive useful reading suggestions, but the measured threshold policy sends 100% of paragraphs to review. Managers should not plan staff reductions or claim time savings from this test; a supervised workflow study must measure time per correctly reviewed paragraph, missed-risk rates, disagreement resolution and queue size. The workload statement follows `tables/routing_test.csv`; the workflow study is a proposed way to assess business value.

**Companies described by the disclosures** could be underrepresented when opportunity paragraphs are missed, or wrongly associated with risk when neutral process descriptions are flagged. The opportunity recall of 27/51 and the 23 neutral→risk errors establish that these classifications occur; no company-level reputational effect has been measured. A reviewer should see the original passage and label definition, and a company-facing use would need an avenue for correction rather than a model-generated ranking being treated as an adjudication.

**Investment, procurement or policy decision-makers and the people affected by their decisions** could be misled if a writing-category label is mistaken for a verified environmental outcome. The dataset measures disclosure wording, and `test:0194` shows a risk-labelled cancelled project receiving an opportunity score of 0.811. That example justifies checking source context; it does not establish the actual environmental outcome or suitability of any investment. A human decision owner and independent factual evidence are therefore proposed process safeguards, not conclusions supplied by this classifier.

### 10.3 Top misuse risks and matched safeguards
The table describes **potential misuse**, motivated by observed errors or the source's scope. None of these harms has been measured in a live deployment.

| Potential misuse and supporting evidence | Technical mitigation to test | Process mitigation / accountable owner | Check before or during use |
|---|---|---|---|
| Treating neutral as safe and automatically discarding a paragraph: 15/106 risk paragraphs are missed | Require review for low-margin or disputed inputs; keep source text and all class scores visible | Analyst signs off exclusions; no unattended exclusion of risk candidates | Risk recall and false-negative counts on a fresh labelled holdout; review sampled exclusions |
| Treating opportunity as a verified sustainability rating: only 27/46 opportunity predictions match labels | Present paragraph-level category and provenance; avoid an unsupported company-level score | Decision owner verifies factual claims independently; provide a correction route | Opportunity precision/recall and audits of downstream use; count disputed ratings |
| Trusting a large model score as certainty: wrong opportunity score 0.811 on `test:0194` | Compare validation-only score calibration and optional withholding of automatic labels; retain fail-closed routing until gates pass | Reviewers see the uncertainty caveat and examples of confident errors | Retained accuracy **with support/intervals**, coverage and high-score error counts on new data |
| Assuming improvements apply to all language patterns: jargon risk recall is 12/14 vs 13/14 | Evaluate class × jargon / forward-looking / truncation slices; test heading consistency | Domain reviewer resolves mixed framing and disputed labels | Publish slice support and confusion counts; investigate material supported deterioration |
| Claiming report-wide truth or generalizing to new issuers/languages: only English paragraphs were tested | Test complete-report segmentation and test set from companies or reporting periods not seen during developments; detect changed input lengths/vocabulary | Keep scope explicit; domain owner approves any scope expansion | New-domain labelled performance, input-drift signal and review workload; pause unsupported extensions |

**Technical priorities** should begin with the opportunity/neutral and neutral/risk boundary data, because those are the largest primary-transformer error families (24 and 23 rows). Compare double-reviewed training examples and richer context with the same primary/secondary metrics; do not select the winning approach on this inspected test. Heading-consistency and longer-context tests address documented sensitivities, while calibration/abstention tests address the gap between model scores and acceptable review workload. These changes are proposals; the error and slice evidence motivates them but does not predict their gains.

**Process priorities** are to keep a human review owner, document source-label disagreements, and evaluate the actual reading workflow before granting greater reliance. A proposed monthly labelled audit should report class/slice support, uncertainty and reviewer decisions, with a sampling budget approved by the team. More automation should require a new test set from companies or reporting periods not seen during development to meet the configured gates and a workflow trial showing benefits without an unacceptable increase in missed risks. This decision follows the failed balanced-quality and routing gates; the monthly cadence is a recommendation rather than a validated optimum.

## 11. Decision and guardrails
The final choice applies the proposed gates from Section 1. **Human review remains mandatory during the pilot**, including retained predictions, until a new issuer/temporal holdout and operating targets are approved. Thresholding is an evaluated candidate mitigation; it is not permission to make consequential decisions automatically.

| Named risk / observed limitation | Feasible mitigation | Evaluation / monitoring check | Decision constraint |
|---|---|---|---|
| Missed risk paragraphs (15 of 106 for transformer seed 42) | Re-annotate mixed-framing training examples with two reviewers; add only reviewed examples to training | Risk recall, risk→neutral counts, reviewer disagreement on a new issuer holdout | No unattended exclusion of potential risks; pause if risk recall <0.85 |
| High model score on an actual error (Section 9) | Review routing and local explanation for disputed cases | Validation-fixed coverage/retained accuracy, high-score error rate; target retained accuracy ≥0.95 | Treat scores as ranking signals; if gates fail, review all |
| Truncation / missing conclusions | Test 256 tokens or paragraph chunking on development data; retain tail context | Subword truncation fraction and truncated/non-truncated risk recall | No claim that 128 tokens covers a complete report; change only after a new evaluation |
| Shared issuers, boilerplate, and near duplicates | Deduplicate exact text; commission issuer-disjoint and time-disjoint sets | Hash overlap, nearest-neighbor similarity, performance on new issuers/years | Random-split scores do not justify new-issuer deployment |
| Domain shift, sparse acronyms, future-goal wording | Label new-domain samples; report jargon and forward-looking slices with support | Slice errors, review rate, input length/jargon drift; proposed monthly labelled audit; approve its sample budget and report support/intervals | Restrict to English climate paragraphs; new language/domain needs validation |
| Misuse as a truth / greenwashing / investment assessment | Show original paragraph, label definition, evidence links and human sign-off | Review decisions and audit output usage; incident count | Classification must not determine investments or environmental compliance |
| Dataset use restrictions | Preserve attribution and source license; resolve permitted use before a commercial pilot | Check dataset/model terms for intended use | Academic evidence does not grant commercial use rights |

**Proposed improvement, not a demonstrated gain:** use the most frequent source→prediction pair and its saved row IDs to create a double-reviewed training annotation guide. Evaluate whether the resulting examples improve that class recall and macro F1 on a fresh holdout while preserving the risk-recall gate. If long/truncated failures dominate, compare 128 vs 256 tokens on validation before the next held-out evaluation. Do not assert that augmentation, longer context, or domain pretraining will improve results until tested.

In [ ]:
D=CONFIG['decision']
def passes_quality(row):return row.macro_f1>=D['min_macro_f1'] and row.risk_recall>=D['min_risk_recall']
base_resource=resources.set_index('model').loc['Selected classical']
base_pass=passes_quality(base) and base_resource.inference_ms_per_paragraph<=D['max_inference_ms_per_paragraph']
choice='Selected classical' if base_pass else 'Human-led review; no model clears all pilot gates'
reasons=[]
if paired_result:
    trans=metric_table.set_index('model').loc['DistilBERT seed 42']
    tr=resources.set_index('model').loc['DistilBERT seed 42']
    transformer_pass=(passes_quality(trans) and trans.risk_recall>=base.risk_recall and
        paired_result['delta_macro_f1']>=D['min_transformer_gain'] and paired_result['ci_low']>0 and
        tr.inference_ms_per_paragraph<=D['max_inference_ms_per_paragraph'])
    if transformer_pass:choice='DistilBERT seed 42 (conditional pilot candidate)'
    reasons.append({'transformer_primary_passes':bool(transformer_pass)})
else:
    reasons.append({'transformer_primary_passes':None,'reason':'controlled GPU evidence pending'})
decision={'choice':choice,'classical_passes':bool(base_pass),'gates':D,'details':reasons,
          'use':'analyst assistance with human review',
          'change_choice_if':'Transformer meets paired-gain, risk-recall and latency gates on controlled data; or a fresh issuer/temporal audit changes performance.',
          'stop_conditions':['Risk recall <0.85 on an adequately supported labelled audit set.',
                             'Domain/language changes without validation.',
                             'Retained accuracy/coverage misses gates: revert to full review.']}
save_json('decision.json',decision)
display(Markdown(f"**Evidence-based recommendation: {choice}.** Classical macro F1 {base.macro_f1:.3f} (target ≥{D['min_macro_f1']:.2f}); risk recall {base.risk_recall:.3f} (target ≥{D['min_risk_recall']:.2f}). {decision['change_choice_if']} Human review remains mandatory for the pilot; no claim of verified environmental performance is permitted."))

## 12. Evidence map and reproducibility record
The checklist links the acceptance questions to concrete outputs. It distinguishes completed analytical evidence from the unavailable local transformer tensors. A completed review run means saved predictions and analyses were recomputed; it does not mean the transformer was retrained. The data and input-run manifests identify which fitted model results are being interpreted.

In [ ]:
audit_rows=[
 ('Environment and traceable run','run_config.json; environment.freeze.txt; source_manifest.json; comparison_source.json in review mode'),
 ('Preprocessing alternative','tables/preprocessing_comparison.csv; Section 4 controlled stop-word comparison'),
 ('Observed failures motivate improvements','tables/qualitative_error_families.csv; tables/qualitative_error_examples.csv; Section 8 proposed experiments'),
 ('Controlled comparison and validity threats','split_manifest.csv; tables/test_metrics.csv; paired_comparison.json; repeated-body and issuer caveats'),
 ('Fine-tuning and constraint trade-offs','recorded trainer_state.json; resource comparison; 57/320 truncation count; seed variation'),
 ('Explainability and limits','explanation_cases.json; exact-margin assertions; token/deletion CSVs; no transformer token claim'),
 ('Triangulated inference','class confusion counts + actual error paragraphs + classical feature contributions; interpretations labelled as hypotheses'),
 ('Conditional decision','decision.json; balanced-quality, risk-recall, latency and routing gates'),
 ('Operational framing','Section 1 proposed targets explicitly labelled assumptions; Section 7.2 explains the missed target'),
 ('Stakeholder explanation','stakeholder_briefing.md; counts, reading consequences and scope limits in Section 10'),
 ('Stakeholder impacts','Section 10.2 ties impacts to missed opportunities, false risk alarms and actual review workload'),
 ('Fairness and misuse safeguards','tables/class_recall_gaps.csv; Section 10.3 technical/process safeguards and checks')]
audit=pd.DataFrame(audit_rows,columns=['criterion','evidence'])
save_table('acceptance_criteria',audit)
display(audit)
save_json('completion.json',{'status':'review complete' if REVIEW_MODE else ('full experiment complete' if transformer_runs else 'baseline complete'),
      'run_id':RUN_ID,'source_comparison_run':REVIEW_DIR.name if REVIEW_MODE else None,
      'transformer_status':transformer_status,'executed_to_final_cell':True,
      'config_sha256':config_hash,'results_table':'tables/test_metrics.csv',
      'training_performed_in_this_execution':bool(RUN_TRANSFORMER),
      'explanation_scope':'classical LR feature contributions and deletion only'})
manifest_files={str(path.relative_to(RUN_DIR)):sha256(path) for path in sorted(RUN_DIR.rglob('*')) if path.is_file() and path.name!='manifest.json'}
save_json('manifest.json',{'run_id':RUN_ID,'sha256':manifest_files})
(ROOT/'artifacts/latest_run.json').write_text(json.dumps({'run_id':RUN_ID,'path':str(RUN_DIR)},indent=2)+'\n')
# A review archive contains analytical evidence, not absent transformer tensor files.
archive_path=shutil.make_archive(str(RUN_DIR),'zip',root_dir=RUN_DIR)
display(Markdown(f'**Evidence run:** `{RUN_ID}` · [configuration]({RUN_DIR.relative_to(ROOT)}/run_config.json) · [metrics]({RUN_DIR.relative_to(ROOT)}/tables/test_metrics.csv) · [checksums]({RUN_DIR.relative_to(ROOT)}/manifest.json) · [evidence ZIP](artifacts/{RUN_ID}.zip)'))
print('Execution:',MODE,'| training performed:',bool(RUN_TRANSFORMER),'| explanation scope: classical model')